In [ ]:
# Cell 1: Load Data and Apply Timestamp Offsets

import pandas as pd
import os

# Define the base path to the data directory relative to the notebook location
data_path = os.path.join('..', 'data')

# Define file names
price_files = {
    2: os.path.join(data_path, 'prices_round_5_day_2.csv'),
    3: os.path.join(data_path, 'prices_round_5_day_3.csv'),
    4: os.path.join(data_path, 'prices_round_5_day_4.csv')
}

trade_files = {
    2: os.path.join(data_path, 'trades_round_5_day_2.csv'),
    3: os.path.join(data_path, 'trades_round_5_day_3.csv'),
    4: os.path.join(data_path, 'trades_round_5_day_4.csv')
}

# Define timestamp offsets (Day 3: +1M, Day 4: +2M)
timestamp_offsets = {
    2: 0,
    3: 1_000_000,
    4: 2_000_000
}

# --- Load and Process Price Data ---
all_prices_list = []
for day, file_path in price_files.items():
    try:
        prices_day_df = pd.read_csv(file_path, sep=';')
        offset = timestamp_offsets[day]
        if offset > 0:
            prices_day_df['timestamp'] += offset
        all_prices_list.append(prices_day_df)
    except FileNotFoundError:
        print(f"Warning: Price file not found at {file_path}")
    except Exception as e:
        print(f"Warning: Error loading price file {file_path}: {e}")

# Concatenate all price data
if all_prices_list:
    df_prices = pd.concat(all_prices_list, ignore_index=True)
else:
    df_prices = pd.DataFrame() # Create empty df if loading failed


# --- Load and Process Trade Data ---
all_trades_list = []
for day, file_path in trade_files.items():
    try:
        trades_day_df = pd.read_csv(file_path, sep=';')
        offset = timestamp_offsets[day]
        if offset > 0:
            trades_day_df['timestamp'] += offset
        all_trades_list.append(trades_day_df)
    except FileNotFoundError:
        print(f"Warning: Trade file not found at {file_path}")
    except Exception as e:
        print(f"Warning: Error loading trade file {file_path}: {e}")

# Concatenate all trade data
if all_trades_list:
    df_trades = pd.concat(all_trades_list, ignore_index=True)
else:
    df_trades = pd.DataFrame() # Create empty df if loading failed



print("Trades Data Head (df_trades):")
if not df_trades.empty:
    print(df_trades.head())
else:
    print("Trade data is empty.")

# ***** ADD THESE LINES AT THE END OF CELL 1 *****
print("\nColumns in df_prices after loading:")
print(df_prices.columns)
print("\nHead of df_prices after loading:")
print(df_prices.head())
# ***** END OF ADDED LINES *****

In [ ]:
# Cell 2: Identify Unique Traders

print("Unique Buyers:", sorted(df_trades['buyer'].unique()))
print("Unique Sellers:", sorted(df_trades['seller'].unique()))

Interesting, Olga only sells

In [ ]:
# Cell 3: Calculate PnL, Trade Count, Buy/Sell Qty, Open/Close Prices per Asset

import pandas as pd

# --- Configuration ---
trader_to_analyze = 'Camilla' # <--- Ensure this trader exists and traded
# ---

pnl_df = pd.DataFrame() # Initialize pnl_df as an empty DataFrame

# Ensure df_prices and df_trades are available from previous cells
if 'df_prices' not in locals() or 'df_trades' not in locals():
    print("Error: df_prices or df_trades not defined. Please run Cell 1 first.")
else:
    pnl_results = {}

    # --- Pre-calculate First and Last Mid-Prices ---
    if df_prices.empty:
        print("Warning: df_prices is empty. Cannot determine closing prices.")
        first_mid_prices = pd.Series(dtype=float)
        last_mid_prices = pd.Series(dtype=float)
        first_overall_timestamp = "N/A"
        last_overall_timestamp = "N/A"
    else:
        # Sort prices by timestamp ONCE
        df_prices_sorted = df_prices.sort_values('timestamp')

        # Get FIRST mid_price for each symbol
        first_prices_df = df_prices_sorted.groupby('product').head(1) # Get the first row for each product
        if 'product' in first_prices_df.columns and 'mid_price' in first_prices_df.columns:
            first_mid_prices = first_prices_df.set_index('product')['mid_price'].fillna(0)
            first_overall_timestamp = df_prices_sorted['timestamp'].min()
            print(f"Using opening prices from timestamp starting at: {first_overall_timestamp}")
        else:
            print("CRITICAL ERROR: 'product' or 'mid_price' column missing when calculating first prices!")
            first_mid_prices = pd.Series(dtype=float)
            first_overall_timestamp = "N/A"


        # Get LAST mid_price for each symbol
        last_prices_df = df_prices_sorted.groupby('product').tail(1) # Get the last row for each product
        if 'product' in last_prices_df.columns and 'mid_price' in last_prices_df.columns:
            last_mid_prices = last_prices_df.set_index('product')['mid_price'].fillna(0)
            last_overall_timestamp = df_prices_sorted['timestamp'].max()
            print(f"Using closing prices from timestamp up to: {last_overall_timestamp}")
        else:
             print("CRITICAL ERROR: 'product' or 'mid_price' column missing when calculating last prices!")
             last_mid_prices = pd.Series(dtype=float)
             last_overall_timestamp = "N/A"


    # --- Filter Trades for the Chosen Trader ---
    if all(col in df_trades.columns for col in ['symbol', 'buyer', 'seller']):
         trader_trades = df_trades[(df_trades['buyer'] == trader_to_analyze) |
                                  (df_trades['seller'] == trader_to_analyze)].copy()
    else:
        print("CRITICAL ERROR: Key columns ('symbol', 'buyer', 'seller') missing in df_trades!")
        trader_trades = pd.DataFrame()

    if trader_trades.empty:
        print(f"\nNo trades found for {trader_to_analyze}.")
    else:
        print(f"\nCalculating PnL for {trader_to_analyze}...")
        # --- Calculate PnL per Symbol (from trades) ---
        if 'symbol' not in trader_trades.columns:
             print("CRITICAL ERROR inside loop preparation: 'symbol' column missing in trader_trades!")
        else:
             for symbol in trader_trades['symbol'].unique():
                required_trade_cols = ['symbol', 'buyer', 'seller', 'quantity', 'price']
                if not all(col in trader_trades.columns for col in required_trade_cols):
                    print(f"CRITICAL ERROR inside loop for symbol {symbol}: Key column missing in trader_trades DataFrame!")
                    continue

                symbol_trades = trader_trades[trader_trades['symbol'] == symbol]
                trade_count = len(symbol_trades)

                buys = symbol_trades[symbol_trades['buyer'] == trader_to_analyze]
                total_quantity_bought = buys['quantity'].sum()
                total_cost = (buys['price'] * buys['quantity']).sum()

                sells = symbol_trades[symbol_trades['seller'] == trader_to_analyze]
                total_quantity_sold = sells['quantity'].sum()
                total_revenue = (sells['price'] * sells['quantity']).sum()

                net_position = total_quantity_bought - total_quantity_sold

                # *** Get Open and Closing Prices ***
                open_price = first_mid_prices.get(symbol, 0) # Get the first mid price
                closing_price = last_mid_prices.get(symbol, 0) # Get the last mid price

                inventory_value = net_position * closing_price
                total_pnl = total_revenue - total_cost + inventory_value

                # *** Store open_price along with other results ***
                pnl_results[symbol] = {
                    'symbol': symbol,
                    'total_pnl': total_pnl,
                    'net_position': net_position,
                    'open_price': open_price,       # Added open price
                    'closing_price': closing_price,
                    'trade_count': trade_count,
                    'qty_bought': total_quantity_bought,
                    'qty_sold': total_quantity_sold,
                }
             # *** END OF FOR LOOP ***

        # --- Display Results ---
        if pnl_results:
            print(f"\n--- PnL, Position, and Trade Stats for {trader_to_analyze} by Asset ---")
            pnl_df = pd.DataFrame(list(pnl_results.values()))

            if not pnl_df.empty and 'total_pnl' in pnl_df.columns:
                pnl_df = pnl_df.sort_values('total_pnl', ascending=False)
            elif not pnl_df.empty:
                print("DEBUG WARNING: 'total_pnl' column not found. Skipping sort.")
                pnl_df = pnl_df.sort_values('symbol')

            # Set display options
            pd.options.display.float_format = '{:,.2f}'.format
            pd.set_option('display.max_columns', None)
            pd.set_option('display.width', 2000)

            # *** MODIFIED: Add 'open_price' to the required columns ***
            required_cols = ['symbol', 'total_pnl', 'net_position',
                             'open_price', 'closing_price', # Added open_price
                             'trade_count', 'qty_bought', 'qty_sold']
            missing_cols = [col for col in required_cols if col not in pnl_df.columns]

            if not missing_cols:
                 print("\nFinal Stats Table:")
                 print(pnl_df.reindex(columns=required_cols))
                 pd.reset_option('display.max_columns')
                 pd.reset_option('display.width')
            else:
                 print(f"\nERROR: Cannot print final table. Missing required columns: {missing_cols}")
                 print("Current pnl_df columns:", pnl_df.columns)
                 print("Current pnl_df head:"); print(pnl_df.head())
                 pd.reset_option('display.max_columns')
                 pd.reset_option('display.width')

            # Overall PnL calculation
            if 'total_pnl' in pnl_df.columns:
                overall_pnl = pnl_df['total_pnl'].sum()
                print(f"\n--- Overall Approximate PnL for {trader_to_analyze}: {overall_pnl:,.2f} ---")
            else:
                print("\nCould not calculate overall PnL because 'total_pnl' column is missing.")

        else:
            print(f"Warning: Trades found for {trader_to_analyze}, but PnL calculation resulted in no entries in pnl_results dictionary.")


# --- Final Check before next cell ---
if not pnl_df.empty:
    # Add open_price to the check
    final_check_cols = ['symbol', 'trade_count', 'qty_bought', 'qty_sold', 'open_price']
    missing_final_cols = [col for col in final_check_cols if col not in pnl_df.columns]
    if missing_final_cols:
         print(f"\nCRITICAL WARNING (End of Cell 3): pnl_df created but missing required columns: {missing_final_cols}!")
    else:
        print("\nCell 3 finished. 'pnl_df' seems correctly populated with requested stats columns.")
elif pnl_df.empty:
    print("\nNote (End of Cell 3): pnl_df is empty. Analysis cell (Cell 4/5) might be skipped or show limited info.")

In [ ]:
# Cell 5: Static Plot of Price, Trades, Position, and PnL using Matplotlib

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker # For formatting axes if needed
import pandas as pd
import numpy as np # Needed for np.union1d

# --- Configuration ---
# <<< CHANGE THESE >>>
participant_to_analyze = 'Camilla'
symbol_to_analyze = 'VOLCANIC_ROCK_VOUCHER_9500'
# ---

print(f"Analyzing Participant: {participant_to_analyze}, Symbol: {symbol_to_analyze}")

# --- Input Data Validation ---
# (Keep the existing validation checks)
if 'df_prices' not in locals() or df_prices.empty:
    print("Error: df_prices not found or is empty. Run Cell 1.")
elif 'df_trades' not in locals() or df_trades.empty:
    print("Error: df_trades not found or is empty. Run Cell 1.")
elif not all(col in df_prices.columns for col in ['product', 'timestamp', 'bid_price_1', 'ask_price_1', 'mid_price']):
     print(f"Error: df_prices is missing required columns. Found: {df_prices.columns}")
elif not all(col in df_trades.columns for col in ['symbol', 'timestamp', 'buyer', 'seller', 'price', 'quantity']):
     print(f"Error: df_trades is missing required columns. Found: {df_trades.columns}")
else:
    # --- Filter Data ---
    # (Keep the existing filtering logic)
    symbol_prices = df_prices[df_prices['product'] == symbol_to_analyze].copy()
    symbol_prices = symbol_prices.sort_values('timestamp').drop_duplicates(subset=['timestamp'], keep='last')

    participant_symbol_trades = df_trades[
        (df_trades['symbol'] == symbol_to_analyze) &
        ((df_trades['buyer'] == participant_to_analyze) |
         (df_trades['seller'] == participant_to_analyze))
    ].copy()
    participant_symbol_trades = participant_symbol_trades.sort_values('timestamp')

    # --- Check if data exists after filtering ---
    if symbol_prices.empty:
        print(f"Error: No price data found for symbol '{symbol_to_analyze}'. Cannot generate plot.")
    elif participant_symbol_trades.empty:
        print(f"Notice: No trades found for participant '{participant_to_analyze}' in symbol '{symbol_to_analyze}'. Plotting only prices.")
        pnl_calculated = False
    else:
        # --- Calculate Position and Cash Flow ---
        # (Keep the existing calculation logic)
        participant_symbol_trades['position_change'] = participant_symbol_trades.apply(
            lambda row: row['quantity'] if row['buyer'] == participant_to_analyze else -row['quantity'], axis=1)
        participant_symbol_trades['cumulative_position'] = participant_symbol_trades['position_change'].cumsum()

        participant_symbol_trades['cash_flow'] = participant_symbol_trades.apply(
            lambda row: -row['quantity'] * row['price'] if row['buyer'] == participant_to_analyze else row['quantity'] * row['price'], axis=1)
        participant_symbol_trades['cumulative_cash_flow'] = participant_symbol_trades['cash_flow'].cumsum()

        # --- Create Combined Timeline for PnL Calculation ---
        # (Keep the existing PnL timeline logic)
        all_timestamps = np.union1d(symbol_prices['timestamp'].unique(),
                                   participant_symbol_trades['timestamp'].unique())
        all_timestamps = np.sort(all_timestamps)
        pnl_timeline = pd.DataFrame(index=all_timestamps)
        pnl_timeline.index.name = 'timestamp'
        trades_indexed = participant_symbol_trades.set_index('timestamp')[['cumulative_position', 'cumulative_cash_flow']]
        pnl_timeline = pnl_timeline.join(trades_indexed)
        pnl_timeline['cumulative_position'] = pnl_timeline['cumulative_position'].ffill().fillna(0)
        pnl_timeline['cumulative_cash_flow'] = pnl_timeline['cumulative_cash_flow'].ffill().fillna(0)
        prices_indexed = symbol_prices.set_index('timestamp')[['mid_price']]
        pnl_timeline = pnl_timeline.join(prices_indexed)
        pnl_timeline['mid_price'] = pnl_timeline['mid_price'].ffill()
        first_valid_price_idx = pnl_timeline['mid_price'].first_valid_index()
        if first_valid_price_idx is not None and pd.isna(pnl_timeline.loc[pnl_timeline.index.min(), 'mid_price']):
             first_valid_price = pnl_timeline.loc[first_valid_price_idx, 'mid_price']
             pnl_timeline['mid_price'] = pnl_timeline['mid_price'].fillna(first_valid_price) # Backfill first NaNs
        pnl_timeline['mid_price'] = pnl_timeline['mid_price'].fillna(0) # Fill remaining NaNs

        # --- Calculate Mark-to-Market PnL ---
        pnl_timeline['inventory_value'] = pnl_timeline['cumulative_position'] * pnl_timeline['mid_price']
        pnl_timeline['mtm_pnl'] = pnl_timeline['cumulative_cash_flow'] + pnl_timeline['inventory_value']
        pnl_calculated = True

    # --- Create Static Matplotlib Plot ---
    # Check again if prices exist before creating the plot structure
    if symbol_prices.empty:
        print("Skipping plot creation as no price data is available.")
    else:
        fig, ax = plt.subplots(3, 1, sharex=True, figsize=(15, 10)) # Create 3 subplots

        # == Plot 1: Prices and Trades ==
        ax[0].plot(symbol_prices['timestamp'], symbol_prices['bid_price_1'], label='Best Bid', color='blue', linestyle='-', linewidth=1, alpha=0.8)
        ax[0].plot(symbol_prices['timestamp'], symbol_prices['ask_price_1'], label='Best Ask', color='red', linestyle='-', linewidth=1, alpha=0.8)
        ax[0].plot(symbol_prices['timestamp'], symbol_prices['mid_price'], label='Mid Price', color='grey', linestyle='--', linewidth=1, alpha=0.7)

        # Add Trade Markers only if trades exist
        if not participant_symbol_trades.empty:
            buy_trades = participant_symbol_trades[participant_symbol_trades['buyer'] == participant_to_analyze]
            sell_trades = participant_symbol_trades[participant_symbol_trades['seller'] == participant_to_analyze]
            ax[0].scatter(buy_trades['timestamp'], buy_trades['price'], label=f'{participant_to_analyze} Buys',
                          marker='^', color='lime', edgecolors='black', s=50, zorder=3) # zorder puts markers on top
            ax[0].scatter(sell_trades['timestamp'], sell_trades['price'], label=f'{participant_to_analyze} Sells',
                          marker='v', color='magenta', edgecolors='black', s=50, zorder=3)

        ax[0].set_ylabel('Price')
        ax[0].set_title(f'Market Prices and {participant_to_analyze}\'s Trades for {symbol_to_analyze}')
        ax[0].legend(loc='best')
        ax[0].grid(True, linestyle='--', alpha=0.5)

        # == Plot 2: Position ==
        if not participant_symbol_trades.empty:
             # Use step plot for position changes
             ax[1].step(participant_symbol_trades['timestamp'], participant_symbol_trades['cumulative_position'],
                        where='post', label='Position', color='purple', linewidth=2)
        ax[1].axhline(0, color='grey', linestyle='--', linewidth=1, label='_nolegend_') # Zero line
        ax[1].set_ylabel('Position')
        ax[1].set_title(f'{participant_to_analyze}\'s Position in {symbol_to_analyze}')
        ax[1].grid(True, linestyle='--', alpha=0.5)
        # Add legend only if step was plotted
        if not participant_symbol_trades.empty:
            ax[1].legend(loc='best')


        # == Plot 3: PnL ==
        if pnl_calculated: # Check if PnL data exists
             ax[2].plot(pnl_timeline.index, pnl_timeline['mtm_pnl'], label='MtM PnL', color='orange', linewidth=2)
             ax[2].legend(loc='best') # Add legend only if PnL was plotted
        ax[2].axhline(0, color='grey', linestyle='--', linewidth=1, label='_nolegend_') # Zero line
        ax[2].set_ylabel('MtM PnL')
        ax[2].set_title(f'{participant_to_analyze}\'s MtM PnL in {symbol_to_analyze}')
        ax[2].grid(True, linestyle='--', alpha=0.5)

        # --- Layout Adjustments ---
        ax[2].set_xlabel('Timestamp') # Set x-axis label only on the bottom plot

        # Improve timestamp readability (optional, adjust as needed)
        # Example: Format as numbers with commas
        formatter = mticker.FormatStrFormatter('%d')
        ax[2].xaxis.set_major_formatter(formatter)
        # Or rotate labels if they overlap
        # plt.setp(ax[2].get_xticklabels(), rotation=30, ha='right')

        plt.tight_layout() # Adjust spacing to prevent labels overlapping
        plt.show() # Display the plot

In [ ]:
# Cell 6: Data Preparation - Merge Trades with State & EXTENDED Future Changes + Volumes

import pandas as pd
import numpy as np

# --- Configuration ---
# Define standard horizons - MUST INCLUDE ALL START/END POINTS needed for Cell 14
horizons_to_calculate = [
    100, 200, 500, 1000, 2500, # Original shorter/medium horizons
    5000, 10000, 25000, 50000   # New longer horizons
]
# Define delayed window (optional, calculated if base horizons exist)
# This specific calculation isn't strictly needed if Cell 14 does it, but harmless to keep
delayed_start_horizon = 100
delayed_end_horizon = 200
# ---

# --- Input Validation ---
# (Keep validation checks as before, ensuring bid/ask/vol exist)
required_dfs = {
    'df_prices': ['product', 'timestamp', 'mid_price', 'bid_price_1', 'ask_price_1', 'bid_volume_1', 'ask_volume_1'],
    'df_trades': ['symbol', 'timestamp', 'buyer', 'seller', 'price', 'quantity']
}
data_valid = True
print("--- Running Data Validation for Cell 6 (Extended Horizons) ---")
for df_name, cols in required_dfs.items():
    if df_name not in locals(): print(f"Error: DataFrame '{df_name}' not found."); data_valid = False; break
    current_df = locals()[df_name]
    if not isinstance(current_df, pd.DataFrame): print(f"Error: Variable '{df_name}' not DataFrame."); data_valid = False; break
    missing_cols = [col for col in cols if col not in current_df.columns]
    if missing_cols: print(f"Error: DataFrame '{df_name}' missing columns: {missing_cols}."); data_valid = False; break
print("--- Data Validation Finished ---")

# --- Main Processing Logic ---
if data_valid:
    print(f"\nPreparing data for future horizons up to {max(horizons_to_calculate)} timestamps.")
    print("Warning: Long horizons will reduce data significantly after NaN drop.")

    # --- 1. Prepare Price Data ---
    print(" - Preparing price data...")
    price_cols = ['timestamp', 'product', 'mid_price', 'bid_price_1', 'ask_price_1', 'bid_volume_1', 'ask_volume_1']
    prices_f = df_prices[price_cols].copy()
    prices_f = prices_f.sort_values(by=['product', 'timestamp'])

    # --- Calculate future changes for EACH horizon ---
    future_change_cols_mid = []
    future_change_cols_bid = []
    future_change_cols_ask = []
    calculated_horizons = [] # Track successful calculations

    for horizon in horizons_to_calculate:
        if horizon % 100 != 0: print(f"   - Skipping horizon {horizon} (not multiple of 100)"); continue
        shift_amount = -int(horizon / 100)
        if shift_amount == 0: print(f"   - Skipping horizon {horizon} (shift is 0)"); continue

        print(f"   - Calculating horizon: {horizon} (shift amount: {shift_amount})")
        calculated_horizons.append(horizon)
        gb = prices_f.groupby('product')

        # Mid, Bid, Ask Price Changes
        for price_type in ['mid_price', 'bid_price_1', 'ask_price_1']:
            col_prefix = price_type.split('_')[0] # mid, bid, ask
            fc_col = f'future_price_change_{col_prefix}_{horizon}'
            prices_f[fc_col] = gb[price_type].shift(shift_amount) - prices_f[price_type]
            # Store column names
            if col_prefix == 'mid': future_change_cols_mid.append(fc_col)
            elif col_prefix == 'bid': future_change_cols_bid.append(fc_col)
            elif col_prefix == 'ask': future_change_cols_ask.append(fc_col)

    # --- Calculate DELAYED future change (Optional - using Mid) ---
    delayed_col_name = None
    if delayed_start_horizon in calculated_horizons and delayed_end_horizon in calculated_horizons:
        print(f"   - Calculating delayed mid-price horizon: {delayed_start_horizon} -> {delayed_end_horizon}")
        start_col = f'future_price_change_mid_{delayed_start_horizon}'
        end_col = f'future_price_change_mid_{delayed_end_horizon}'
        delayed_col_name = f'delayed_change_mid_{delayed_start_horizon}_{delayed_end_horizon}'
        prices_f[delayed_col_name] = prices_f[end_col] - prices_f[start_col]
        print(f"     -> Added column: {delayed_col_name}")
    else:
        print(f"   - Skipping delayed mid-price calculation ({delayed_start_horizon}-{delayed_end_horizon}) - base horizons missing.")

    prices_f = prices_f.rename(columns={'product': 'symbol'})

    # Select columns for merging
    merge_cols = ['timestamp', 'symbol', 'mid_price', 'bid_price_1', 'ask_price_1', 'bid_volume_1', 'ask_volume_1'] \
                 + future_change_cols_mid + future_change_cols_bid + future_change_cols_ask
    if delayed_col_name and delayed_col_name in prices_f.columns: merge_cols.append(delayed_col_name)
    merge_cols = [col for col in merge_cols if col in prices_f.columns] # Ensure existence
    prices_f_merged = prices_f[merge_cols].sort_values(by='timestamp')
    print(f"   - Price data prepared. Shape: {prices_f_merged.shape}")

    # --- 2. Prepare Trade Data ---
    print(" - Preparing trade data...")
    trades_m = df_trades.copy().sort_values(by='timestamp')
    print(f"   - Trade data prepared. Shape: {trades_m.shape}")

    # --- 3. Merge ---
    print(" - Merging trades with prices...")
    df_merged = pd.merge_asof(
        trades_m, prices_f_merged, on='timestamp', by='symbol', direction='backward')
    print(f"   - Merge attempted. Shape before NaN drop: {df_merged.shape}")

    # --- 4. Clean up ---
    initial_rows = len(df_merged)
    # Define subset for dropping NaNs: need current prices/VOLUMES + ALL calculated future changes
    dropna_subset = ['mid_price', 'bid_price_1', 'ask_price_1', 'bid_volume_1', 'ask_volume_1'] \
                    + [col for col in future_change_cols_mid if col in df_merged.columns] \
                    + [col for col in future_change_cols_bid if col in df_merged.columns] \
                    + [col for col in future_change_cols_ask if col in df_merged.columns]
    if delayed_col_name and delayed_col_name in df_merged.columns: dropna_subset.append(delayed_col_name)
    print(f"   - Dropping rows with NaNs in subset (checking {len(dropna_subset)} columns)")
    actual_dropna_subset = [col for col in dropna_subset if col in df_merged.columns]
    if actual_dropna_subset:
        df_merged = df_merged.dropna(subset=actual_dropna_subset)
    else: print("   - Warning: No valid columns found for dropna subset.")
    removed_rows = initial_rows - len(df_merged)
    print(f"   - Merge results cleaned. Final shape: {df_merged.shape}") # <<< Check this shape!
    if removed_rows > 0: print(f"   - Note: Removed {removed_rows} rows (expected due to long horizons).")

    # --- 5. Add Spread Feature ---
    if 'bid_price_1' in df_merged.columns and 'ask_price_1' in df_merged.columns:
         print(" - Calculating spread...")
         df_merged['spread'] = df_merged['ask_price_1'] - df_merged['bid_price_1']

    # --- 6. Display Result ---
    if not df_merged.empty:
        print("\n--- Merged DataFrame Info (Extended Horizons) ---")
        df_merged.info()
    else: print("\n--- Merged DataFrame is Empty after cleaning ---")

else:
    print("\nSkipping Cell 6 due to validation errors.")

# --- Final Debug Check ---
# (Keep the final debug check block exactly as it was)
print("\n--- DEBUG: Final check of df_merged at end of Cell 6 ---")
if 'df_merged' in locals():
    print(f"Variable 'df_merged' EXISTS.")
    if isinstance(df_merged, pd.DataFrame):
        print(f"Type is DataFrame. Is it empty? {df_merged.empty}")
        if not df_merged.empty:
            print(f"Shape: {df_merged.shape}") # <<< Check this shape!
            print(f"Columns: {df_merged.columns.tolist()}")
            print("Head of final df_merged:")
            with pd.option_context('display.max_columns', None, 'display.width', 2000): print(df_merged.head())
        else: print("Reason for empty df_merged: Likely extensive NaNs dropped due to long horizons.")
    else: print(f"Variable 'df_merged' exists BUT IS NOT A DATAFRAME. Type: {type(df_merged)}")
else: print("CRITICAL ERROR: Variable 'df_merged' DOES NOT EXIST at end of Cell 6.")
print("--- End of Cell 6 Final Check ---")

In [ ]:
# Cell 8: Visualize Self-Trading Events (Caesar->Caesar) - Plotly Interactive

import plotly.graph_objects as go
from plotly.subplots import make_subplots # Not strictly needed for 1 plot, but good practice
import pandas as pd

# --- Configuration ---
participant_to_analyze = 'Camilla'
symbol_to_analyze = 'VOLCANIC_ROCK_VOUCHER_9500'
# ---

print(f"Visualizing self-trades for Participant: {participant_to_analyze}, Symbol: {symbol_to_analyze}")

# --- Input Validation ---
data_valid = True
if 'df_prices' not in locals() or df_prices.empty:
    print("Error: df_prices not found or is empty. Run Cell 1.")
    data_valid = False
elif 'df_merged' not in locals() or df_merged.empty:
    print("Error: df_merged not found or is empty. Run Cell 6 first.")
    data_valid = False
elif not all(col in df_prices.columns for col in ['product', 'timestamp', 'mid_price', 'bid_price_1', 'ask_price_1']):
     print(f"Error: df_prices is missing required columns.")
     data_valid = False
elif not all(col in df_merged.columns for col in ['symbol', 'timestamp', 'buyer', 'seller', 'price', 'quantity']):
     print(f"Error: df_merged is missing required columns.")
     data_valid = False

if data_valid:
    # --- Filter Data ---
    symbol_prices = df_prices[df_prices['product'] == symbol_to_analyze].copy()
    symbol_prices = symbol_prices.sort_values('timestamp')

    self_trades = df_merged[
        (df_merged['symbol'] == symbol_to_analyze) &
        (df_merged['buyer'] == participant_to_analyze) &
        (df_merged['seller'] == participant_to_analyze)
    ].copy()
    self_trades = self_trades.sort_values('timestamp')

    # --- Check if data exists after filtering ---
    if symbol_prices.empty:
        print(f"Error: No price data found for symbol '{symbol_to_analyze}'. Cannot generate plot.")
    elif self_trades.empty:
        print(f"Notice: No self-trades found for participant '{participant_to_analyze}' in symbol '{symbol_to_analyze}'. Plotting only prices.")
    else:
         print(f"Found {len(self_trades)} self-trades for {participant_to_analyze} in {symbol_to_analyze}.")

    # --- Create Interactive Plot ---
    if not symbol_prices.empty:
        # Using make_subplots even for one plot allows easy extension later if needed
        fig = make_subplots(rows=1, cols=1,
                          subplot_titles=(f'Mid Price and {participant_to_analyze} Self-Trades for {symbol_to_analyze}',))

        # == Plot Prices ==
        fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['mid_price'], mode='lines',
                                 name='Mid Price', line=dict(color='grey', width=1), opacity=0.8), row=1, col=1)
        # Optional: Add Bid/Ask
        # fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['bid_price_1'], mode='lines',
        #                          name='Best Bid', line=dict(color='blue', width=0.5), opacity=0.5), row=1, col=1)
        # fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['ask_price_1'], mode='lines',
        #                          name='Best Ask', line=dict(color='red', width=0.5), opacity=0.5), row=1, col=1)


        # == Plot Self-Trade Markers ==
        if not self_trades.empty:
            fig.add_trace(go.Scatter(x=self_trades['timestamp'], y=self_trades['price'], mode='markers',
                                     marker=dict(color='orange', size=7, symbol='circle', line=dict(color='black', width=1)),
                                     name=f'{participant_to_analyze} Self-Trades ({len(self_trades)})',
                                     hovertemplate='<b>Self-Trade</b><br>Timestamp: %{x}<br>Price: %{y:,.2f}<br>Quantity: %{customdata[0]}<extra></extra>',
                                     customdata=self_trades[['quantity']]), row=1, col=1)

        # --- Layout Updates ---
        fig.update_layout(
            title_text=f'Analysis for Participant: {participant_to_analyze} | Symbol: {symbol_to_analyze}', # Overall title
            height=600,
            xaxis_title="Timestamp",
            yaxis_title="Price",
            legend_title_text="Traces",
            hovermode='x unified' # Show hover info for all traces at a timestamp
        )

        # Optional: Add range slider for easier navigation on dense plots
        # fig.update_xaxes(rangeslider_visible=True)

        fig.show()

else:
    print("\nSkipping Cell 8 due to missing input DataFrames or columns.")

In [ ]:
# Cell 7: Analyze Bot Interactions vs. MARKET Baselines (Simplified Aggregation)

import pandas as pd
import numpy as np

# --- Configuration ---
symbol_to_analyze = 'VOLCANIC_ROCK_VOUCHER_9500' # <<< Choose the asset symbol to analyze
# Define the horizons that were calculated in Cell 6
horizons_to_calculate = [100, 500, 1000, 2500, 5000, 10000]
min_event_trades = 5 # Define minimum trades for analysis (can be adjusted or removed)
# ---

# --- Input Validation ---
valid_input = True
base_future_cols_needed = []
for h in horizons_to_calculate: base_future_cols_needed.append(f'future_price_change_mid_{h}') # Assuming mid price

if 'df_merged' not in locals() or df_merged.empty: print(f"Error: df_merged not found."); valid_input = False
elif 'df_prices' not in locals() or df_prices.empty: print(f"Error: df_prices not found."); valid_input = False
else:
    missing_base_cols = [col for col in base_future_cols_needed if col not in df_merged.columns]
    if missing_base_cols: print(f"Error: df_merged missing base columns: {missing_base_cols}"); valid_input = False
    elif not all(col in df_prices.columns for col in ['product', 'timestamp', 'mid_price']): print(f"Error: df_prices missing required columns."); valid_input = False
    else: print("Input data validated.")

# --- Helper: T-Stat vs Zero ---
# (Keep helper function as before)
def calculate_t_stat_vs_zero(mean, std, n):
    if pd.isna(mean) or pd.isna(std) or std == 0 or n < 2: return np.nan
    se = std / np.sqrt(n); t_stat = mean / se
    return t_stat

# --- Main Analysis Loop ---
if valid_input:
    print(f"\nAnalyzing Bot interactions for Symbol: {symbol_to_analyze}")
    print(f"Analyzing across horizons: {horizons_to_calculate}")
    print(f"Comparing against MARKET baseline.")

    # --- 1. Calculate MARKET Baselines ---
    # (Keep baseline calculation logic as before)
    print(f"\nCalculating MARKET Baseline Future Changes ({symbol_to_analyze})...")
    market_baselines_directional = {}
    market_baselines_absolute = {}
    df_prices_symbol = df_prices[df_prices['product'] == symbol_to_analyze].copy()
    if df_prices_symbol.empty:
         print(f"Error: No price data found for {symbol_to_analyze} in df_prices.")
         for h in horizons_to_calculate: market_baselines_directional[h] = np.nan; market_baselines_absolute[h] = np.nan
    else:
        df_prices_symbol = df_prices_symbol.sort_values(by=['timestamp'])
        print("Horizon | Mean Change (Drift) | Mean Abs Change (Magnitude) | Std Dev Change | Min Change | Max Change")
        print("--------|---------------------|-----------------------------|----------------|------------|------------")
        for h in horizons_to_calculate:
            if h % 100 != 0: continue
            shift_amount = -int(h / 100)
            if shift_amount == 0: continue
            future_price = df_prices_symbol['mid_price'].shift(shift_amount)
            price_change = future_price - df_prices_symbol['mid_price']
            mean_change = price_change.mean(); mean_abs_change = price_change.abs().mean()
            std_dev_change = price_change.std(); min_change = price_change.min(); max_change = price_change.max()
            market_baselines_directional[h] = mean_change; market_baselines_absolute[h] = mean_abs_change
            print(f"{h:>7} | {mean_change:>19,.2f} | {mean_abs_change:>27,.2f} | {std_dev_change:>14,.2f} | {min_change:>10,.2f} | {max_change:>10,.2f}")

    # --- 2. Filter TRADE Data ---
    df_symbol_trades = df_merged[df_merged['symbol'] == symbol_to_analyze].copy()
    if df_symbol_trades.empty:
        print(f"\nNo trade data found for symbol '{symbol_to_analyze}' in df_merged.")
    else:
        # --- 3. Feature Engineering ---
        df_symbol_trades['buyer_seller_pair'] = df_symbol_trades['buyer'] + '->' + df_symbol_trades['seller']

        # --- 4. Define Columns to Aggregate ---
        # Use the base future price change columns (assuming mid price)
        cols_to_aggregate = [f'future_price_change_mid_{h}' for h in horizons_to_calculate if f'future_price_change_mid_{h}' in df_symbol_trades.columns]
        if not cols_to_aggregate:
             print("Error: No valid future price change columns found in df_symbol_trades to aggregate.")
        else:
            all_results = [] # Store results dictionaries here
            participants = sorted(list(pd.unique(df_symbol_trades[['buyer', 'seller']].values.ravel('K'))))

            # --- 5. Inner loops: Participant Actions & Pairs ---
            def process_group(group_df, group_id_info):
                """ Helper to process aggregation and store results """
                trade_count = len(group_df)
                if trade_count >= min_event_trades: # Apply min trades filter here
                    try:
                        # Calculate mean and std directly for the relevant columns
                        means = group_df[cols_to_aggregate].mean()
                        stds = group_df[cols_to_aggregate].std()

                        # Combine into a dictionary
                        stats = {'trade_count': trade_count}
                        stats.update({f'avg_{col}': means[col] for col in cols_to_aggregate if col in means}) # Add avg_ prefix
                        stats.update({f'std_{col}': stds[col] for col in cols_to_aggregate if col in stds})   # Add std_ prefix
                        stats.update(group_id_info) # Add symbol, type, pair info
                        all_results.append(stats)
                    except Exception as e:
                         print(f"    Error during aggregation for {group_id_info}: {e}. Skipping group.")

            print("\n - Analyzing interaction pairs, buyers, sellers...")
            # Analyze Buyer->ALL
            for participant in participants:
                event_df = df_symbol_trades[df_symbol_trades['buyer'] == participant]
                process_group(event_df, {'symbol': symbol, 'analysis_type': 'Buyer->ALL', 'participant_or_pair': f"{participant}->ALL"})
            # Analyze ALL->Seller
            for participant in participants:
                event_df = df_symbol_trades[df_symbol_trades['seller'] == participant]
                process_group(event_df, {'symbol': symbol, 'analysis_type': 'ALL->Seller', 'participant_or_pair': f"ALL->{participant}"})
            # Analyze Pairs/Self-Trades
            pair_groups = df_symbol_trades.groupby(['buyer', 'seller'], observed=True)
            for (buyer, seller), event_df in pair_groups:
                 analysis_type = 'Self-Trade' if buyer == seller else 'Pair'
                 process_group(event_df, {'symbol': symbol, 'analysis_type': analysis_type, 'participant_or_pair': f"{buyer}->{seller}"})

            # --- 6. Convert results to DataFrame ---
            print("\nAnalysis loops complete. Creating final DataFrame...")
            if not all_results:
                print("No events met the minimum trade criteria.")
                all_stats_df = pd.DataFrame()
            else:
                all_stats_df = pd.DataFrame(all_results)
                print(f"Created results DataFrame with {len(all_stats_df)} rows.")

                # --- 7. Calculate Difference from Baseline & T-Stats ---
                id_cols = ['symbol', 'participant_or_pair', 'analysis_type', 'trade_count']
                final_stat_cols = []
                for h in horizons_to_calculate:
                    base_col = f'future_price_change_mid_{h}' # Original column name
                    avg_col = f'avg_{base_col}' # Column name after aggregation
                    std_col = f'std_{base_col}' # Column name after aggregation
                    diff_col = f'diff_vs_baseline_{h}'
                    t_stat_col = f't_stat_{h}' # T-stat for this horizon

                    if avg_col in all_stats_df.columns:
                        # Calculate Diff vs Baseline
                        baseline = market_baselines_directional.get(h, np.nan)
                        if pd.notna(baseline):
                            all_stats_df[diff_col] = all_stats_df[avg_col] - baseline
                        else:
                            all_stats_df[diff_col] = np.nan

                        # Calculate T-Stat vs Zero
                        if std_col in all_stats_df.columns:
                             all_stats_df[t_stat_col] = all_stats_df.apply(
                                 lambda row: calculate_t_stat_vs_zero(row[avg_col], row[std_col], row['trade_count']), axis=1)
                             final_stat_cols.extend([avg_col, std_col, diff_col, t_stat_col]) # Add all calculated stats
                        else:
                             final_stat_cols.extend([avg_col, diff_col]) # Add avg and diff even if std/tstat failed
                             print(f"Warning: Std column {std_col} missing for T-stat calc.")
                    else:
                         print(f"Warning: Avg column {avg_col} missing.")


                all_display_cols = id_cols + sorted(list(set(final_stat_cols))) # Use set to remove duplicates, then sort
                all_display_cols = [c for c in all_display_cols if c in all_stats_df.columns] # Final check
                all_stats_df = all_stats_df[all_display_cols]

                # --- 8. Display Results ---
                if all_stats_df.empty:
                    print("\nNo statistics calculated (all_stats DataFrame is empty).")
                else:
                    print(f"\n--- Combined Interaction Statistics vs. MARKET Baseline ({symbol_to_analyze}) ---")
                    # Rank by first horizon's t-stat
                    rank_horizon = horizons_to_calculate[0]
                    rank_col = f't_stat_{rank_horizon}'
                    if rank_col in all_stats_df.columns:
                        all_stats_df[rank_col] = all_stats_df[rank_col].fillna(0) # Fill NaN for sorting
                        ranked_df = all_stats_df.sort_values(by=rank_col, key=abs, ascending=False, na_position='last')
                    else:
                        print(f"Warning: Sort column '{rank_col}' not found.")
                        ranked_df = all_stats_df # Keep original order

                    # Select columns for final display (e.g., avg, diff, tstat for first two horizons)
                    print_cols_subset = id_cols[:]
                    rename_map = {}
                    for h in horizons_to_calculate[:2]: # Show first two horizons for brevity
                         avg_col = f'avg_future_price_change_mid_{h}'
                         diff_col = f'diff_vs_baseline_{h}'
                         tstat_col = f't_stat_{h}'
                         if avg_col in ranked_df.columns: print_cols_subset.append(avg_col); rename_map[avg_col] = f'Avg_{h}'
                         if diff_col in ranked_df.columns: print_cols_subset.append(diff_col); rename_map[diff_col] = f'Diff_{h}'
                         if tstat_col in ranked_df.columns: print_cols_subset.append(tstat_col); rename_map[tstat_col] = f'T_{h}'

                    ranked_df_display = ranked_df[print_cols_subset].rename(columns=rename_map)
                    final_print_cols = [rename_map.get(c, c) for c in print_cols_subset]

                    print(f"\nDEBUG: Displaying columns: {final_print_cols}")
                    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.2f}'.format):
                        print("\nFinal Statistics Table (Showing Top Rows):")
                        print(ranked_df_display.head(50)) # Show more rows

                    print("\n--- Interpretation Notes ---")
                    # (Keep interpretation notes as before)

In [ ]:
# Cell 8: Visualize Self-Trading Events (Caesar->Caesar)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# --- Configuration ---
participant_to_analyze = 'Caesar'
symbol_to_analyze = 'VOLCANIC_ROCK_VOUCHER_1000'
# ---

print(f"Visualizing self-trades for Participant: {participant_to_analyze}, Symbol: {symbol_to_analyze}")

# --- Input Validation ---
# Ensure df_prices and df_merged exist and have necessary columns
data_valid = True
if 'df_prices' not in locals() or df_prices.empty:
    print("Error: df_prices not found or is empty. Run Cell 1.")
    data_valid = False
elif 'df_merged' not in locals() or df_merged.empty:
    print("Error: df_merged not found or is empty. Run Cell 6 first.")
    data_valid = False
elif not all(col in df_prices.columns for col in ['product', 'timestamp', 'mid_price', 'bid_price_1', 'ask_price_1']):
     print(f"Error: df_prices is missing required columns.")
     data_valid = False
elif not all(col in df_merged.columns for col in ['symbol', 'timestamp', 'buyer', 'seller', 'price', 'quantity']):
     print(f"Error: df_merged is missing required columns.")
     data_valid = False

if data_valid:
    # --- Filter Data ---
    # Price data for the symbol
    symbol_prices = df_prices[df_prices['product'] == symbol_to_analyze].copy()
    symbol_prices = symbol_prices.sort_values('timestamp')

    # Self-Trades for the specific participant and symbol
    self_trades = df_merged[
        (df_merged['symbol'] == symbol_to_analyze) &
        (df_merged['buyer'] == participant_to_analyze) &
        (df_merged['seller'] == participant_to_analyze) # Condition for self-trade
    ].copy()
    # Ensure sorting if not already guaranteed by df_merged structure
    self_trades = self_trades.sort_values('timestamp')

    # --- Check if data exists after filtering ---
    if symbol_prices.empty:
        print(f"Error: No price data found for symbol '{symbol_to_analyze}'. Cannot generate plot.")
    elif self_trades.empty:
        print(f"Notice: No self-trades found for participant '{participant_to_analyze}' in symbol '{symbol_to_analyze}'. Plotting only prices.")
    else:
         print(f"Found {len(self_trades)} self-trades for {participant_to_analyze} in {symbol_to_analyze}.")

    # --- Create Static Matplotlib Plot ---
    if not symbol_prices.empty:
        fig, ax = plt.subplots(1, 1, figsize=(15, 6)) # Single plot

        # == Plot Prices ==
        ax.plot(symbol_prices['timestamp'], symbol_prices['mid_price'], label='Mid Price', color='grey', linestyle='-', linewidth=1, alpha=0.8)
        # Optional: Plot Bid/Ask as well for context (can make it noisy)
        # ax.plot(symbol_prices['timestamp'], symbol_prices['bid_price_1'], label='Best Bid', color='blue', linestyle='-', linewidth=0.5, alpha=0.5)
        # ax.plot(symbol_prices['timestamp'], symbol_prices['ask_price_1'], label='Best Ask', color='red', linestyle='-', linewidth=0.5, alpha=0.5)

        # == Plot Self-Trade Markers ==
        if not self_trades.empty:
            # Use a distinct marker for self-trades
            # Size marker by quantity? (Needs scaling) scale_factor = 0.1; sizes = self_trades['quantity'] * scale_factor
            ax.scatter(self_trades['timestamp'], self_trades['price'],
                       label=f'{participant_to_analyze} Self-Trades ({len(self_trades)})',
                       marker='o', color='orange', edgecolors='black', s=40, zorder=3) # Circle marker

        ax.set_ylabel('Price')
        ax.set_xlabel('Timestamp')
        ax.set_title(f'Mid Price and {participant_to_analyze} Self-Trades for {symbol_to_analyze}')
        ax.legend(loc='best')
        ax.grid(True, linestyle='--', alpha=0.5)

        # Format X-axis
        formatter = mticker.FormatStrFormatter('%d')
        ax.xaxis.set_major_formatter(formatter)

        plt.tight_layout()
        plt.show()

else:
    print("\nSkipping Cell 8 due to missing input DataFrames or columns.")

In [ ]:
# Cell 8: Visualize Specific Buyer->Seller Trades (Plotly Interactive - Generalized)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# --- Configuration ---
# <<< SET THE SYMBOL, BUYER, AND SELLER TO ANALYZE >>>
symbol_to_analyze = 'VOLCANIC_ROCK_VOUCHER_10500' # Example symbol
buyer_to_analyze = 'Caesar'      # Set the specific buyer (e.g., 'Caesar', 'Paris')
seller_to_analyze = 'Caesar'     # Set the specific seller (e.g., 'Caesar', 'Camilla')
# Note: Setting buyer=seller allows visualizing self-trades
# ---

print(f"Visualizing trades for Symbol: {symbol_to_analyze}")
print(f"Specific Interaction: Buyer='{buyer_to_analyze}' -> Seller='{seller_to_analyze}'")


# --- Input Validation ---
data_valid = True
if 'df_prices' not in locals() or df_prices.empty:
    print("Error: df_prices not found or is empty. Run Cell 1.")
    data_valid = False
elif 'df_merged' not in locals() or df_merged.empty:
    print("Error: df_merged not found or is empty. Run Cell 6 first.")
    data_valid = False
elif not all(col in df_prices.columns for col in ['product', 'timestamp', 'mid_price', 'bid_price_1', 'ask_price_1']):
     print(f"Error: df_prices is missing required columns.")
     data_valid = False
elif not all(col in df_merged.columns for col in ['symbol', 'timestamp', 'buyer', 'seller', 'price', 'quantity']):
     print(f"Error: df_merged is missing required columns.")
     data_valid = False

if data_valid:
    # --- Filter Data ---
    symbol_prices = df_prices[df_prices['product'] == symbol_to_analyze].copy()
    symbol_prices = symbol_prices.sort_values('timestamp')

    # --- Filter for the SPECIFIC buyer -> seller pair ---
    pair_trades = df_merged[
        (df_merged['symbol'] == symbol_to_analyze) &
        (df_merged['buyer'] == buyer_to_analyze) &    # Match specific buyer
        (df_merged['seller'] == seller_to_analyze)   # Match specific seller
    ].copy()
    pair_trades = pair_trades.sort_values('timestamp')

    # --- Check if data exists after filtering ---
    if symbol_prices.empty:
        print(f"Error: No price data found for symbol '{symbol_to_analyze}'. Cannot generate plot.")
    elif pair_trades.empty:
        print(f"Notice: No trades found for Buyer='{buyer_to_analyze}' -> Seller='{seller_to_analyze}' in symbol '{symbol_to_analyze}'. Plotting only prices.")
    else:
         print(f"Found {len(pair_trades)} trades for {buyer_to_analyze}->{seller_to_analyze} in {symbol_to_analyze}.")

    # --- Create Interactive Plot ---
    if not symbol_prices.empty:
        fig = make_subplots(rows=1, cols=1,
                            subplot_titles=(f'Mid Price and {buyer_to_analyze}->{seller_to_analyze} Trades for {symbol_to_analyze}',))

        # == Plot Prices ==
        fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['mid_price'], mode='lines',
                                 name='Mid Price', line=dict(color='grey', width=1), opacity=0.8), row=1, col=1)
        # Optional: Add Bid/Ask
        fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['bid_price_1'], mode='lines', name='Best Bid', line=dict(color='blue', width=0.5), opacity=0.5), row=1, col=1)
        fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['ask_price_1'], mode='lines', name='Best Ask', line=dict(color='red', width=0.5), opacity=0.5), row=1, col=1)


        # == Plot Specific Pair Trade Markers ==
        if not pair_trades.empty:
            pair_label = f'{buyer_to_analyze}->{seller_to_analyze} ({len(pair_trades)})'
            fig.add_trace(go.Scatter(x=pair_trades['timestamp'], y=pair_trades['price'], mode='markers',
                                     marker=dict(color='orange', size=7, symbol='circle', line=dict(color='black', width=1)),
                                     name=pair_label,
                                     hovertemplate=f'<b>{buyer_to_analyze}->{seller_to_analyze}</b><br>Timestamp: %{{x}}<br>Price: %{{y:,.2f}}<br>Quantity: %{{customdata[0]}}<extra></extra>',
                                     customdata=pair_trades[['quantity']]), row=1, col=1)

        # --- Layout Updates ---
        fig.update_layout(
            title_text=f'Interaction Analysis: {buyer_to_analyze} -> {seller_to_analyze} | Symbol: {symbol_to_analyze}',
            height=600,
            xaxis_title="Timestamp",
            yaxis_title="Price",
            legend_title_text="Traces",
            hovermode='x unified'
        )

        # Optional: Add range slider
        # fig.update_xaxes(rangeslider_visible=True)

        fig.show() # Display the plot

else:
    print("\nSkipping Cell 8 due to missing input DataFrames or columns.")

In [ ]:
# Cell 9: Systematic Analysis of Bot Behavior Impact (Corrected Column Name)

import pandas as pd
import numpy as np
# Required for t-statistic (or use math for simpler version)
from scipy import stats

# --- Configuration ---
# Horizons used in Cell 6
horizons_to_calculate = [100, 500, 1000, 2500, 5000, 10000]
min_event_trades = 5 # Minimum trades for an event to be analyzed
symbols_to_exclude = ['RAINFOREST_RESIN'] # Updated exclusion list
# ---

# --- Input Validation ---
if 'df_merged' not in locals() or df_merged.empty:
    print("Error: df_merged not found or is empty. Run Cell 6 first.")
    valid_input = False
elif 'df_prices' not in locals() or df_prices.empty:
    print("Error: df_prices not found or is empty. Run Cell 1 first.")
    valid_input = False
else:
    # Check if base future price change columns exist
    base_future_cols_needed = [f'future_price_change_mid_{h}' for h in horizons_to_calculate]
    missing_base_cols = [col for col in base_future_cols_needed if col not in df_merged.columns]
    if missing_base_cols:
        print(f"Error: df_merged missing base columns: {missing_base_cols}")
        valid_input = False
    else:
        valid_input = True

# --- Helper Function: Calculate Stats for an Event ---
def analyze_event(df_event_trades, market_baselines_stats, horizons, event_name="Event"):
    """
    Calculates statistics for future price changes following specific trade events.
    Uses 'future_price_change_mid_{h}' columns.
    """
    trade_count = len(df_event_trades)
    if trade_count < min_event_trades:
        return None
    event_stats = {"trade_count": trade_count}
    for h in horizons:
        # --- >>> CORRECTED COLUMN NAME <<< ---
        col = f'future_price_change_mid_{h}' # Use the mid-price change column from df_merged
        # --- >>> END CORRECTION <<< ---

        if col not in df_event_trades.columns:
            # This check prevents errors if a specific horizon wasn't calculated in Cell 6
            print(f"Warning: Column {col} not found in event data for {event_name}. Skipping horizon {h}.")
            continue

        event_mean = df_event_trades[col].mean()
        event_std = df_event_trades[col].std()

        # Store stats using the horizon number for consistency in output df
        event_stats[f'avg_change_{h}'] = event_mean
        event_stats[f'std_dev_change_{h}'] = event_std

        # Get baseline stats for this horizon
        baseline_mean = market_baselines_stats.get(h, {}).get('mean', np.nan)
        # baseline_std = market_baselines_stats.get(h, {}).get('std', np.nan) # Needed for 2-sample t-test
        # baseline_count = market_baselines_stats.get(h, {}).get('count', 0) # Needed for 2-sample t-test

        # Calculate difference
        if pd.notna(baseline_mean):
            event_stats[f'diff_vs_baseline_{h}'] = event_mean - baseline_mean
        else:
            event_stats[f'diff_vs_baseline_{h}'] = np.nan

        # Calculate t-statistic vs baseline mean
        if pd.notna(event_std) and event_std > 0 and trade_count > 1 and pd.notna(baseline_mean):
             se_event = event_std / np.sqrt(trade_count)
             # Avoid division by zero if se_event is somehow zero
             if se_event > 1e-9: # Add small tolerance check
                 t_stat = (event_mean - baseline_mean) / se_event
                 event_stats[f't_stat_{h}'] = t_stat
             else:
                 event_stats[f't_stat_{h}'] = np.nan # Indeterminate t-stat
        else:
             event_stats[f't_stat_{h}'] = np.nan # Cannot calculate t-stat

    return event_stats


# --- Main Analysis Loop ---
if valid_input:
    all_results = []
    participants = sorted(list(pd.unique(df_merged[['buyer', 'seller']].values.ravel('K'))))
    symbols = sorted(df_merged['symbol'].unique())
    print(f"Initial symbols found: {len(symbols)}")
    symbols = [s for s in symbols if s not in symbols_to_exclude]
    print(f"Excluding symbols: {symbols_to_exclude}")
    print(f"Analyzing {len(symbols)} symbols and {len(participants)} participants...")

    # --- Outer loop: Symbols ---
    for i, symbol in enumerate(symbols):
        print(f"  Analyzing Symbol {i+1}/{len(symbols)}: {symbol}")
        df_symbol_trades = df_merged[df_merged['symbol'] == symbol].copy()
        # Use 'product' column for df_prices
        df_prices_symbol = df_prices[df_prices['product'] == symbol].copy()

        if df_symbol_trades.empty or df_prices_symbol.empty:
            print(f"    Skipping {symbol} due to missing trade or price data.")
            continue

        # --- Calculate Market Baselines (Mean, StdDev, Count) ---
        market_baselines_stats = {}
        df_prices_symbol = df_prices_symbol.sort_values(by=['timestamp'])
        for h in horizons_to_calculate:
            if h % 100 != 0: continue
            shift_amount = -int(h / 100)
            if shift_amount == 0: continue
            # Ensure mid_price exists before shifting
            if 'mid_price' in df_prices_symbol.columns:
                 future_price = df_prices_symbol['mid_price'].shift(shift_amount)
                 price_change = future_price - df_prices_symbol['mid_price']
                 baseline_mean = price_change.mean()
                 baseline_std = price_change.std()
                 baseline_count = price_change.count()
                 market_baselines_stats[h] = {'mean': baseline_mean, 'std': baseline_std, 'count': baseline_count}
            else:
                 print(f"    Warning: mid_price column missing for baseline calc in {symbol}")
                 market_baselines_stats[h] = {'mean': np.nan, 'std': np.nan, 'count': 0}


        # --- Inner loops: Participant Actions & Pairs ---
        # (Keep inner loop logic exactly as it was, calling the corrected analyze_event)
        for participant in participants:
            event_df = df_symbol_trades[df_symbol_trades['buyer'] == participant]
            stats = analyze_event(event_df, market_baselines_stats, horizons_to_calculate, event_name=f"{participant}->ALL")
            if stats:
                 stats['symbol'] = symbol; stats['analysis_type'] = 'Buyer->ALL'; stats['participant_or_pair'] = f"{participant}->ALL"
                 all_results.append(stats)

            event_df = df_symbol_trades[df_symbol_trades['seller'] == participant]
            stats = analyze_event(event_df, market_baselines_stats, horizons_to_calculate, event_name=f"ALL->{participant}")
            if stats:
                 stats['symbol'] = symbol; stats['analysis_type'] = 'ALL->Seller'; stats['participant_or_pair'] = f"ALL->{participant}"
                 all_results.append(stats)

        pair_groups = df_symbol_trades.groupby(['buyer', 'seller'], observed=True)
        for (buyer, seller), event_df in pair_groups:
             analysis_type = 'Self-Trade' if buyer == seller else 'Pair'
             stats = analyze_event(event_df, market_baselines_stats, horizons_to_calculate, event_name=f"{buyer}->{seller}")
             if stats:
                 stats['symbol'] = symbol; stats['analysis_type'] = analysis_type; stats['participant_or_pair'] = f"{buyer}->{seller}"
                 all_results.append(stats)

    # --- Convert results to DataFrame ---
    print("\nAnalysis loops complete. Creating final DataFrame...")
    if not all_results:
        print("No events met the minimum trade criteria across analyzed symbols.")
        all_results_df = pd.DataFrame()
    else:
        all_results_df = pd.DataFrame(all_results)
        print(f"Created results DataFrame with {len(all_results_df)} rows.")

        # --- Post-processing & Ranking (Example) ---
        id_cols = ['symbol', 'participant_or_pair', 'analysis_type', 'trade_count']
        # Dynamically find all stat columns generated
        stat_cols = sorted([col for col in all_results_df.columns if col not in id_cols])
        all_results_df = all_results_df[id_cols + stat_cols] # Reorder

        rank_horizon = 100
        rank_col = f't_stat_{rank_horizon}'
        if rank_col in all_results_df.columns:
            print(f"\n--- Top Potential Signals (Excluding {symbols_to_exclude}) (Ranked by abs({rank_col})) ---")
            # Fill NaNs in rank column before sorting to avoid errors
            all_results_df[rank_col] = all_results_df[rank_col].fillna(0)
            ranked_df = all_results_df.sort_values(by=rank_col, key=lambda col: col.abs(), ascending=False, na_position='last')

            # Select columns to display for ranking
            display_cols = id_cols + [f'avg_change_{rank_horizon}', f'diff_vs_baseline_{rank_horizon}', rank_col]
            # Also show another horizon for comparison
            compare_horizon = 1000
            compare_cols = [f'avg_change_{compare_horizon}', f'diff_vs_baseline_{compare_horizon}', f't_stat_{compare_horizon}']
            # Add compare columns only if they exist
            display_cols.extend([c for c in compare_cols if c in ranked_df.columns])

            # Ensure display_cols only contains columns that actually exist in ranked_df
            display_cols = [c for c in display_cols if c in ranked_df.columns]

            with pd.option_context('display.max_rows', 150, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.2f}'.format):
                print(ranked_df[display_cols].head(150)) # Show more rows
        else:
            print(f"Cannot rank by {rank_col}, column not found.")
            print("Printing unsorted head:")
            with pd.option_context('display.max_rows', 20, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.2f}'.format):
                 print(all_results_df.head())


        # (Keep Next Steps printout as it was)
        print("\n--- Next Steps ---")
        print(" - Examine the ranked table for high absolute t-stats and meaningful diff_vs_baseline.")
        print(" - Further filter/sort by 'trade_count' for more confidence.")
        print(" - Analyze consistency across different horizons for the top signals.")
        print(" - Perform visual checks (Cell 8) for the most interesting findings.")

else:
     print("Skipping Cell 9 due to missing input DataFrames or validation errors.")

In [ ]:
# Cell 10: Analyze Specific Buyer Activity within a Time Window

import pandas as pd
import numpy as np
# from scipy import stats # Keep if using scipy's ttest_ind

# --- Configuration ---
TARGET_BUYER = 'Pablo'
START_TIMESTAMP = 2_000_000 # Start of Day 3
END_TIMESTAMP = 3_000_000   # Start of Day 4 (exclusive)
# Horizons used in Cell 6
horizons_to_calculate = [100, 500, 1000, 2500, 5000, 10000]
min_event_trades_window = 3 # Minimum trades for the buyer in this window to analyze
# ---

# --- Input Validation ---
if 'df_merged' not in locals() or df_merged.empty:
    print("Error: df_merged not found or is empty. Run Cell 6 first.")
    valid_input = False
elif 'df_prices' not in locals() or df_prices.empty:
    print("Error: df_prices not found or is empty. Run Cell 1 first.")
    valid_input = False
else:
    valid_input = True

# --- Helper Function (Reusing from Cell 9) ---
# (Ensure the analyze_event function from Cell 9 is available or copy it here)
# For clarity, let's copy it here, assuming it's needed standalone
def analyze_event_window(df_event_trades, market_baselines_stats, horizons):
    trade_count = len(df_event_trades)
    # Use the window-specific minimum trades
    if trade_count < min_event_trades_window:
        return None
    event_stats = {"trade_count": trade_count}
    for h in horizons:
        col = f'future_price_change_{h}'
        if col not in df_event_trades.columns: continue
        event_mean = df_event_trades[col].mean()
        event_std = df_event_trades[col].std()
        event_stats[f'avg_change_{h}'] = event_mean
        event_stats[f'std_dev_change_{h}'] = event_std
        baseline_mean = market_baselines_stats.get(h, {}).get('mean', np.nan)
        # baseline_std = market_baselines_stats.get(h, {}).get('std', np.nan) # Needed for 2-sample t-test
        # baseline_count = market_baselines_stats.get(h, {}).get('count', 0) # Needed for 2-sample t-test
        if pd.notna(baseline_mean):
            event_stats[f'diff_vs_baseline_{h}'] = event_mean - baseline_mean
        else:
            event_stats[f'diff_vs_baseline_{h}'] = np.nan
        # Simplified t-stat comparing event mean to baseline mean
        if pd.notna(event_std) and event_std > 0 and trade_count > 1 and pd.notna(baseline_mean):
             se_event = event_std / np.sqrt(trade_count)
             t_stat = (event_mean - baseline_mean) / se_event
             event_stats[f't_stat_{h}'] = t_stat
        else:
             event_stats[f't_stat_{h}'] = np.nan
    return event_stats

# --- Main Analysis Logic ---
if valid_input:
    print(f"Analyzing BUY activity for '{TARGET_BUYER}' between timestamps {START_TIMESTAMP} and {END_TIMESTAMP} (exclusive)")

    # --- 1. Filter TRADE data for the buyer and time window ---
    buyer_trades_window = df_merged[
        (df_merged['buyer'] == TARGET_BUYER) &
        (df_merged['timestamp'] >= START_TIMESTAMP) &
        (df_merged['timestamp'] < END_TIMESTAMP)
    ].copy()

    if buyer_trades_window.empty:
        print(f"No trades found for buyer '{TARGET_BUYER}' in the specified time window.")
    else:
        print(f"Found {len(buyer_trades_window)} trades for '{TARGET_BUYER}' in the window.")
        # --- 2. Filter PRICE data for the time window ---
        prices_window = df_prices[
            (df_prices['timestamp'] >= START_TIMESTAMP) &
            (df_prices['timestamp'] < END_TIMESTAMP)
        ].copy()

        if prices_window.empty:
            print("Error: No price data found for the specified time window. Cannot calculate baseline.")
        else:
            print(f"Using {len(prices_window)} price records from the window for baseline calculation.")
            # --- 3. Iterate through symbols TRADED BY BUYER in the window ---
            symbols_traded_in_window = sorted(buyer_trades_window['symbol'].unique())
            print(f"Analyzing {len(symbols_traded_in_window)} symbols traded by {TARGET_BUYER} in window: {symbols_traded_in_window}")

            window_results = []
            for i, symbol in enumerate(symbols_traded_in_window):
                print(f"  Analyzing Symbol {i+1}/{len(symbols_traded_in_window)}: {symbol}")

                # Filter event trades for this symbol (already filtered by buyer & time)
                event_df = buyer_trades_window[buyer_trades_window['symbol'] == symbol]

                # Filter prices for this symbol and time window
                df_prices_symbol_window = prices_window[prices_window['product'] == symbol].copy()

                if df_prices_symbol_window.empty:
                    print(f"    Skipping {symbol}: No price data in window.")
                    continue

                # --- Calculate Market Baselines for this symbol IN THIS WINDOW ---
                market_baselines_stats_window = {}
                df_prices_symbol_window = df_prices_symbol_window.sort_values(by=['timestamp'])
                valid_baseline = False
                for h in horizons_to_calculate:
                    if h % 100 != 0: continue
                    shift_amount = -int(h / 100)
                    if shift_amount == 0: continue

                    # Calculate future change based *only* on prices within the window
                    future_price = df_prices_symbol_window['mid_price'].shift(shift_amount)
                    price_change = future_price - df_prices_symbol_window['mid_price']
                    baseline_mean = price_change.mean()
                    baseline_std = price_change.std()
                    baseline_count = price_change.count()
                    market_baselines_stats_window[h] = {'mean': baseline_mean, 'std': baseline_std, 'count': baseline_count}
                    if baseline_count > 0: valid_baseline = True # Mark if any baseline was calculable

                if not valid_baseline:
                     print(f"    Skipping {symbol}: Could not calculate valid baseline stats in window.")
                     continue

                # --- Analyze the event (Pablo's buys in window) ---
                stats = analyze_event_window(event_df, market_baselines_stats_window, horizons_to_calculate)

                if stats:
                     stats['symbol'] = symbol
                     stats['analysis_type'] = f"{TARGET_BUYER}->ALL (Window)"
                     stats['participant_or_pair'] = f"{TARGET_BUYER}->ALL" # Keep consistent naming if needed
                     window_results.append(stats)
                     print(f"    Analyzed {stats['trade_count']} trades. Avg Change@{horizons_to_calculate[0]}: {stats.get(f'avg_change_{horizons_to_calculate[0]}', 'N/A'):.2f}")
                else:
                     print(f"    Skipping {symbol}: Not enough trades ({len(event_df)}) by {TARGET_BUYER} in window (min: {min_event_trades_window}).")


            # --- 4. Display Results for the Window ---
            if not window_results:
                print(f"\nNo symbols met the minimum trade criteria for {TARGET_BUYER} in this window.")
            else:
                window_results_df = pd.DataFrame(window_results)
                print(f"\nCreated results DataFrame with {len(window_results_df)} rows for the time window.")

                # Reorder columns
                id_cols = ['symbol', 'participant_or_pair', 'analysis_type', 'trade_count']
                stat_cols = [col for col in window_results_df.columns if col not in id_cols]
                window_results_df = window_results_df[id_cols + sorted(stat_cols)]

                # Rank by t-statistic magnitude for a specific horizon
                rank_horizon = 100
                rank_col = f't_stat_{rank_horizon}'
                if rank_col in window_results_df.columns:
                    print(f"\n--- Potential Signals for {TARGET_BUYER} Buys (Window: {START_TIMESTAMP}-{END_TIMESTAMP}) ---")
                    print(f"--- (Ranked by abs(t-statistic) for Horizon {rank_horizon}) ---")
                    ranked_df = window_results_df.sort_values(by=rank_col, key=lambda col: col.abs(), ascending=False, na_position='last')

                    display_cols = id_cols + [f'avg_change_{rank_horizon}', f'diff_vs_baseline_{rank_horizon}', rank_col]
                    compare_horizon = 1000
                    compare_cols = [f'avg_change_{compare_horizon}', f'diff_vs_baseline_{compare_horizon}', f't_stat_{compare_horizon}']
                    display_cols.extend([c for c in compare_cols if c in ranked_df.columns])

                    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.2f}'.format):
                        print(ranked_df.reindex(columns=display_cols)) # Use reindex for safety
                else:
                    print(f"Cannot rank by {rank_col}, column not found. Printing unsorted:")
                    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.2f}'.format):
                         print(window_results_df)

else:
     print("Skipping Cell 10 due to missing input DataFrames.")

In [ ]:
# Cell 8: Visualize Specific Buyer->Seller Trades (Plotly Interactive - Generalized)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# --- Configuration ---
# <<< SET THE SYMBOL, BUYER, AND SELLER TO ANALYZE >>>
symbol_to_analyze = 'PICNIC_BASKET1' # Example symbol
buyer_to_analyze = 'Camilla'      # Set the specific buyer (e.g., 'Caesar', 'Paris')
seller_to_analyze = 'Pablo'     # Set the specific seller (e.g., 'Caesar', 'Camilla')
# Note: Setting buyer=seller allows visualizing self-trades
# ---

print(f"Visualizing trades for Symbol: {symbol_to_analyze}")
print(f"Specific Interaction: Buyer='{buyer_to_analyze}' -> Seller='{seller_to_analyze}'")


# --- Input Validation ---
data_valid = True
if 'df_prices' not in locals() or df_prices.empty:
    print("Error: df_prices not found or is empty. Run Cell 1.")
    data_valid = False
elif 'df_merged' not in locals() or df_merged.empty:
    print("Error: df_merged not found or is empty. Run Cell 6 first.")
    data_valid = False
elif not all(col in df_prices.columns for col in ['product', 'timestamp', 'mid_price', 'bid_price_1', 'ask_price_1']):
     print(f"Error: df_prices is missing required columns.")
     data_valid = False
elif not all(col in df_merged.columns for col in ['symbol', 'timestamp', 'buyer', 'seller', 'price', 'quantity']):
     print(f"Error: df_merged is missing required columns.")
     data_valid = False

if data_valid:
    # --- Filter Data ---
    symbol_prices = df_prices[df_prices['product'] == symbol_to_analyze].copy()
    symbol_prices = symbol_prices.sort_values('timestamp')

    # --- Filter for the SPECIFIC buyer -> seller pair ---
    pair_trades = df_merged[
        (df_merged['symbol'] == symbol_to_analyze) &
        (df_merged['buyer'] == buyer_to_analyze) &    # Match specific buyer
        (df_merged['seller'] == seller_to_analyze)   # Match specific seller
    ].copy()
    pair_trades = pair_trades.sort_values('timestamp')

    # --- Check if data exists after filtering ---
    if symbol_prices.empty:
        print(f"Error: No price data found for symbol '{symbol_to_analyze}'. Cannot generate plot.")
    elif pair_trades.empty:
        print(f"Notice: No trades found for Buyer='{buyer_to_analyze}' -> Seller='{seller_to_analyze}' in symbol '{symbol_to_analyze}'. Plotting only prices.")
    else:
         print(f"Found {len(pair_trades)} trades for {buyer_to_analyze}->{seller_to_analyze} in {symbol_to_analyze}.")

    # --- Create Interactive Plot ---
    if not symbol_prices.empty:
        fig = make_subplots(rows=1, cols=1,
                            subplot_titles=(f'Mid Price and {buyer_to_analyze}->{seller_to_analyze} Trades for {symbol_to_analyze}',))

        # == Plot Prices ==
        fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['mid_price'], mode='lines',
                                 name='Mid Price', line=dict(color='grey', width=1), opacity=0.8), row=1, col=1)
        # Optional: Add Bid/Ask
        fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['bid_price_1'], mode='lines', name='Best Bid', line=dict(color='blue', width=0.5), opacity=0.5), row=1, col=1)
        fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['ask_price_1'], mode='lines', name='Best Ask', line=dict(color='red', width=0.5), opacity=0.5), row=1, col=1)


        # == Plot Specific Pair Trade Markers ==
        if not pair_trades.empty:
            pair_label = f'{buyer_to_analyze}->{seller_to_analyze} ({len(pair_trades)})'
            fig.add_trace(go.Scatter(x=pair_trades['timestamp'], y=pair_trades['price'], mode='markers',
                                     marker=dict(color='orange', size=7, symbol='circle', line=dict(color='black', width=1)),
                                     name=pair_label,
                                     hovertemplate=f'<b>{buyer_to_analyze}->{seller_to_analyze}</b><br>Timestamp: %{{x}}<br>Price: %{{y:,.2f}}<br>Quantity: %{{customdata[0]}}<extra></extra>',
                                     customdata=pair_trades[['quantity']]), row=1, col=1)

        # --- Layout Updates ---
        fig.update_layout(
            title_text=f'Interaction Analysis: {buyer_to_analyze} -> {seller_to_analyze} | Symbol: {symbol_to_analyze}',
            height=600,
            xaxis_title="Timestamp",
            yaxis_title="Price",
            legend_title_text="Traces",
            hovermode='x unified'
        )

        # Optional: Add range slider
        # fig.update_xaxes(rangeslider_visible=True)

        fig.show() # Display the plot

else:
    print("\nSkipping Cell 8 due to missing input DataFrames or columns.")

In [ ]:
# Cell 13: Analyze DELAYED Price Change (t+100 to t+200) After Target Buyer (Corrected Col Names)

import pandas as pd
import numpy as np
# from scipy import stats # Optional

# --- Configuration ---
TARGET_BUYER = 'Pablo'
DELAYED_WINDOW_START = 100
DELAYED_WINDOW_END = 200
min_trades_per_symbol = 5
# ---

# --- Input Validation ---
# --- >>> USE CORRECT COLUMN NAMES <<< ---
start_change_col = f'future_price_change_mid_{DELAYED_WINDOW_START}' # Added _mid_
end_change_col = f'future_price_change_mid_{DELAYED_WINDOW_END}'     # Added _mid_
# --- >>> END CORRECTION <<< ---
delayed_col_name = f'delayed_change_{DELAYED_WINDOW_START}_{DELAYED_WINDOW_END}' # Keep generic name for calculated column

valid_input = True
if 'df_merged' not in locals() or df_merged.empty:
    print(f"Error: df_merged not found or is empty. Run Cell 6 first.")
    valid_input = False
elif start_change_col not in df_merged.columns: # Check for corrected name
    print(f"Error: Required column '{start_change_col}' not found in df_merged.")
    print(f"Ensure Cell 6 included horizon {DELAYED_WINDOW_START} and calculated _mid_ change.")
    valid_input = False
elif end_change_col not in df_merged.columns: # Check for corrected name
    print(f"Error: Required column '{end_change_col}' not found in df_merged.")
    print(f"Ensure Cell 6 included horizon {DELAYED_WINDOW_END} and calculated _mid_ change.")
    valid_input = False
else:
    print(f"Input df_merged found with required columns for delayed analysis: '{start_change_col}', '{end_change_col}'")


# --- Main Analysis Logic ---
if valid_input:
    print(f"\nAnalyzing DELAYED price change (t+{DELAYED_WINDOW_START} to t+{DELAYED_WINDOW_END}) after '{TARGET_BUYER}' buys.")

    # --- 1. Filter TRADE data for the target buyer ---
    buyer_trades = df_merged[df_merged['buyer'] == TARGET_BUYER].copy()

    if buyer_trades.empty:
        print(f"No trades found for buyer '{TARGET_BUYER}'.")
    else:
        print(f"Found {len(buyer_trades)} total buy trades for '{TARGET_BUYER}'.")

        # --- 2. Calculate the DELAYED price change ---
        # Uses the corrected start_col and end_col names
        print(f" - Calculating delayed change column: '{delayed_col_name}'...")
        buyer_trades[delayed_col_name] = buyer_trades[end_col] - buyer_trades[start_col]
        print(f"   - Calculation complete.")

        # --- 3. Group by symbol and calculate stats ---
        # (Aggregation logic remains the same, uses the new delayed_col_name)
        print(f" - Grouping by symbol and calculating average delayed change...")
        delayed_stats = buyer_trades.groupby('symbol').agg(
            trade_count=(delayed_col_name, 'count'),
            avg_delayed_change=(delayed_col_name, 'mean'),
            std_dev_delayed_change=(delayed_col_name, 'std')
        ).reset_index()

        # --- 4. Calculate T-statistic vs Zero ---
        # (T-stat logic remains the same)
        def calculate_t_stat_vs_zero(row):
            mean = row['avg_delayed_change']; std = row['std_dev_delayed_change']; n = row['trade_count']
            if pd.isna(mean) or pd.isna(std) or std == 0 or n < 2: return np.nan
            se = std / np.sqrt(n); t_stat = mean / se
            return t_stat
        delayed_stats['t_stat_vs_zero'] = delayed_stats.apply(calculate_t_stat_vs_zero, axis=1)

        # --- 5. Filter by minimum trades ---
        # (Filtering logic remains the same)
        initial_symbols = len(delayed_stats)
        delayed_stats = delayed_stats[delayed_stats['trade_count'] >= min_trades_per_symbol]
        filtered_symbols = len(delayed_stats)
        print(f"   - Kept {filtered_symbols}/{initial_symbols} symbols with >= {min_trades_per_symbol} trades.")

        # --- 6. Display Results ---
        # (Display logic remains the same)
        if delayed_stats.empty: print(f"\nNo symbols met min trade criteria.")
        else:
            print(f"\n--- Average DELAYED Price Change ({DELAYED_WINDOW_START} to {DELAYED_WINDOW_END} ticks) After {TARGET_BUYER} Buys ---")
            delayed_stats = delayed_stats.sort_values(by='t_stat_vs_zero', key=lambda col: col.abs(), ascending=False, na_position='last')
            print_cols = ['symbol', 'trade_count', 'avg_delayed_change', 'std_dev_delayed_change', 't_stat_vs_zero']
            with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000, 'display.float_format', '{:,.3f}'.format):
                print(delayed_stats.reindex(columns=print_cols))
            # ... (Interpretation notes) ...

else:
     print("Skipping Cell 13 due to validation errors.")

In [ ]:
        # --- Ranking & Display ---
        # Example: Rank by t-statistic magnitude for the shortest delayed window
        rank_window = delayed_windows_to_analyze[0] # e.g., (100, 200)
        rank_col = f't_stat_{rank_window[0]}_{rank_window[1]}'

        if rank_col in all_delayed_results_df.columns:
            print(f"\n--- Top Potential Delayed Signals (Ranked by abs({rank_col})) ---")
            ranked_df = all_delayed_results_df.sort_values(by=rank_col, key=lambda col: col.abs(), ascending=False, na_position='last')

            # --- MODIFIED SECTION TO INCLUDE ALL WINDOWS ---
            # Define columns to display - Start with ID columns
            display_cols_final = id_cols[:] # Copy id_cols

            # Add stats columns for ALL calculated windows
            rename_map = {} # Dictionary for renaming columns
            for start_h, end_h in delayed_windows_to_analyze:
                avg_col = f'avg_delayed_change_{start_h}_{end_h}'
                std_col = f'std_delayed_change_{start_h}_{end_h}'
                tstat_col = f't_stat_{start_h}_{end_h}'

                # Add columns to display list if they exist in the DataFrame
                if avg_col in ranked_df.columns: display_cols_final.append(avg_col)
                # if std_col in ranked_df.columns: display_cols_final.append(std_col) # Optional: Add StdDev back if needed
                if tstat_col in ranked_df.columns: display_cols_final.append(tstat_col)

                # Add entries to rename map for better readability
                if avg_col in ranked_df.columns: rename_map[avg_col] = f'Avg_{start_h}_{end_h}'
                # if std_col in ranked_df.columns: rename_map[std_col] = f'Std_{start_h}_{end_h}'
                if tstat_col in ranked_df.columns: rename_map[tstat_col] = f'T_{start_h}_{end_h}'
            # --- END OF MODIFIED SECTION ---

            # Select only existing columns for display and apply renaming
            ranked_df_display = ranked_df[display_cols_final].rename(columns=rename_map)
            # Get the final list of potentially renamed columns
            final_print_cols = [rename_map.get(c, c) for c in display_cols_final]

            print(f"Displaying columns: {final_print_cols}") # Debug print
            with pd.option_context('display.max_rows', 50, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.3f}'.format):
                print(ranked_df_display[final_print_cols].head(50)) # Print with new column names
        else:
            print(f"Cannot rank by {rank_col}, column not found.")
            # ... (rest of error handling)

In [ ]:
# Cell 14: Systematic Analysis of EXTENDED DELAYED Bot Behavior Impact (Save to CSV)

import pandas as pd
import numpy as np
# from scipy import stats # Optional
import os # To define output path

# --- Configuration ---
delayed_windows_to_analyze = [
    (100, 200), (100, 500), (100, 1000), (100, 2500),
    (100, 5000), (100, 10000), (100, 25000), (100, 50000)
]
min_event_trades = 5
symbols_to_exclude = ['KELP', 'RAINFOREST_RESIN', 'SQUID_INK']
output_filename = "delayed_analysis_results.csv" # <<< Name for the output CSV file
# ---

# --- Input Validation ---
# (Keep validation as before)
valid_input = True
required_base_horizons = set()
for start_h, end_h in delayed_windows_to_analyze: required_base_horizons.add(start_h); required_base_horizons.add(end_h)
if 'df_merged' not in locals() or df_merged.empty: print(f"Error: df_merged not found."); valid_input = False
else:
    missing_base_cols = []
    for h in required_base_horizons:
        col_name = f'future_price_change_mid_{h}'
        if col_name not in df_merged.columns: missing_base_cols.append(col_name)
    if missing_base_cols: print(f"Error: df_merged missing base columns: {missing_base_cols}"); valid_input = False
    else: print("Input df_merged found with required columns.")

# --- Helper: T-Stat vs Zero ---
# (Keep helper function as before)
def calculate_t_stat_vs_zero(mean, std, n):
    if pd.isna(mean) or pd.isna(std) or std == 0 or n < 2: return np.nan
    se = std / np.sqrt(n); t_stat = mean / se
    return t_stat

# --- Main Analysis Loop ---
if valid_input:
    all_delayed_results = []
    participants = sorted(list(pd.unique(df_merged[['buyer', 'seller']].values.ravel('K'))))
    symbols = sorted(df_merged['symbol'].unique())
    symbols = [s for s in symbols if s not in symbols_to_exclude]
    print(f"\nExcluding symbols: {symbols_to_exclude}")
    print(f"Analyzing {len(symbols)} symbols...")

    # --- Outer loop: Symbols ---
    for i, symbol in enumerate(symbols):
        print(f"  Analyzing Symbol {i+1}/{len(symbols)}: {symbol}")
        df_symbol_trades = df_merged[df_merged['symbol'] == symbol].copy()
        if df_symbol_trades.empty: continue

        # --- Calculate Delayed Change Columns ---
        valid_delayed_cols = []
        for start_h, end_h in delayed_windows_to_analyze:
            start_col = f'future_price_change_mid_{start_h}'
            end_col = f'future_price_change_mid_{end_h}'
            delayed_col = f'delayed_change_{start_h}_{end_h}'
            if start_col in df_symbol_trades.columns and end_col in df_symbol_trades.columns:
                df_symbol_trades[delayed_col] = df_symbol_trades[end_col] - df_symbol_trades[start_col]
                valid_delayed_cols.append(delayed_col)
            # else: print(f"    Skipping {delayed_col} - missing base.") # Optional verbose skip message

        if not valid_delayed_cols: continue

        # --- Define columns to aggregate ---
        cols_to_aggregate = valid_delayed_cols

        # --- Inner loops: Participant Actions & Pairs ---
        # (Keep process_group helper and inner loops as before)
        def process_group(group_df, group_id_info):
            if len(group_df) >= min_event_trades:
                try:
                    means = group_df[cols_to_aggregate].mean()
                    stds = group_df[cols_to_aggregate].std()
                    count = len(group_df)
                    stats = {'trade_count': count}
                    stats.update({f'avg_{col}': means[col] for col in cols_to_aggregate})
                    stats.update({f'std_{col}': stds[col] for col in cols_to_aggregate})
                    stats.update(group_id_info)
                    all_delayed_results.append(stats)
                except Exception as e: print(f"    Error aggregation {group_id_info}: {e}.")

        for participant in participants:
            process_group(df_symbol_trades[df_symbol_trades['buyer'] == participant], {'symbol': symbol, 'analysis_type': 'Buyer->ALL', 'participant_or_pair': f"{participant}->ALL"})
            process_group(df_symbol_trades[df_symbol_trades['seller'] == participant], {'symbol': symbol, 'analysis_type': 'ALL->Seller', 'participant_or_pair': f"ALL->{participant}"})
        pair_groups = df_symbol_trades.groupby(['buyer', 'seller'], observed=True)
        for (buyer, seller), event_df in pair_groups:
             analysis_type = 'Self-Trade' if buyer == seller else 'Pair'
             process_group(event_df, {'symbol': symbol, 'analysis_type': analysis_type, 'participant_or_pair': f"{buyer}->{seller}"})

    # --- Convert results to DataFrame ---
    print("\nAnalysis loops complete. Creating final DataFrame...")
    if not all_delayed_results:
        print("No events met the minimum trade criteria across analyzed symbols.")
        all_delayed_results_df = pd.DataFrame()
    else:
        all_delayed_results_df = pd.DataFrame(all_delayed_results)
        print(f"Created results DataFrame with {len(all_delayed_results_df)} rows.")

        # --- Calculate T-Statistics and Finalize Columns ---
        # (Keep t-stat calculation logic as before)
        id_cols = ['symbol', 'participant_or_pair', 'analysis_type', 'trade_count']
        final_stat_cols = []
        for start_h, end_h in delayed_windows_to_analyze:
            delayed_col = f'delayed_change_{start_h}_{end_h}'
            avg_col = f'avg_{delayed_col}'; std_col = f'std_{delayed_col}'; t_stat_col = f't_stat_{start_h}_{end_h}'
            if avg_col in all_delayed_results_df.columns and std_col in all_delayed_results_df.columns:
                all_delayed_results_df[t_stat_col] = all_delayed_results_df.apply( lambda row: calculate_t_stat_vs_zero(row[avg_col], row[std_col], row['trade_count']), axis=1)
                final_stat_cols.extend([avg_col, std_col, t_stat_col])
            # else: print(f"Warning: Skipping t-stat for {delayed_col}.") # Optional verbose skip

        all_display_cols = id_cols + sorted(final_stat_cols)
        all_display_cols = [c for c in all_display_cols if c in all_delayed_results_df.columns]
        # Create the final DataFrame with the desired columns BEFORE saving/displaying
        if all_display_cols:
             all_delayed_results_df = all_delayed_results_df[all_display_cols]
        else:
             print("Warning: No stat columns generated successfully.")


        # --- >>> SAVE TO CSV <<< ---
        if not all_delayed_results_df.empty:
            try:
                # Construct path relative to notebook if needed, or use absolute
                # Assuming notebook is in 'research', save in 'research' folder
                output_path = output_filename
                # Or save outside research: output_path = os.path.join('..', output_filename)
                all_delayed_results_df.to_csv(output_path, index=False, float_format='%.4f') # Save without index, format floats
                print(f"\nSuccessfully saved results to: {os.path.abspath(output_path)}")
            except Exception as e:
                print(f"\nERROR saving results to CSV: {e}")
        # --- >>> END SAVE TO CSV <<< ---


        # --- Ranking & Display ---
        # (Keep ranking and display logic as before)
        rank_window = delayed_windows_to_analyze[0]
        rank_col = f't_stat_{rank_window[0]}_{rank_window[1]}'
        if rank_col in all_delayed_results_df.columns:
            print(f"\n--- Top Potential Delayed Signals (Ranked by abs({rank_col})) ---")
            ranked_df = all_delayed_results_df.sort_values(by=rank_col, key=lambda col: col.abs(), ascending=False, na_position='last')
            display_cols_final = id_cols[:]
            rename_map = {}
            for start_h, end_h in delayed_windows_to_analyze:
                avg_col = f'avg_delayed_change_{start_h}_{end_h}'; tstat_col = f't_stat_{start_h}_{end_h}'
                if avg_col in ranked_df.columns: display_cols_final.append(avg_col)
                if tstat_col in ranked_df.columns: display_cols_final.append(tstat_col)
                if avg_col in ranked_df.columns: rename_map[avg_col] = f'Avg_{start_h}_{end_h}'
                if tstat_col in ranked_df.columns: rename_map[tstat_col] = f'T_{start_h}_{end_h}'

            ranked_df_display = ranked_df[display_cols_final].rename(columns=rename_map)
            final_print_cols = [rename_map.get(c, c) for c in display_cols_final]
            print(f"Displaying columns: {final_print_cols}")
            with pd.option_context('display.max_rows', 50, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.3f}'.format):
                print(ranked_df_display[final_print_cols].head(50))
        else:
            print(f"Cannot rank by {rank_col}, column not found.")
            # ... (rest of error handling)

        # (Keep Next Steps printout)
        print("\n--- Next Steps ---")
        print(" - Examine ranked table for high absolute t-stats in the EXTENDED delayed windows.")
        print(" - Look for consistency across short and long delayed windows.")

else:
     print("Skipping Cell 14 due to validation errors.")

In [ ]:
# Cell 15: Feature Engineering for ML Model (Enhanced + New Features)

import pandas as pd
import numpy as np

# --- Configuration ---
TARGET_HORIZON_START = 100
TARGET_HORIZON_END = 200
# Using delayed mid-price change as target
TARGET_COL_BASE = f'delayed_change_mid_{TARGET_HORIZON_START}_{TARGET_HORIZON_END}'

ROLLING_WINDOW_SHORT = 5
ROLLING_WINDOW_LONG = 20
INTERESTING_PAIRS = [('Caesar', 'Paris'), ('Paris', 'Caesar'), ('Caesar','Caesar')]
# ---

# --- Input Validation ---
valid_input = True
# Add L1 volumes to required cols for df_merged
required_cols = [
    'timestamp', 'symbol', 'buyer', 'seller', 'price', 'quantity',
    'mid_price', 'bid_price_1', 'ask_price_1', 'bid_volume_1', 'ask_volume_1', 'spread',
    # Need the base future changes to calculate the target
    f'future_price_change_mid_{TARGET_HORIZON_START}',
    f'future_price_change_mid_{TARGET_HORIZON_END}'
]
if 'df_merged' not in locals() or df_merged.empty:
    print(f"Error: df_merged not found or is empty. Run Cell 6 (Bid/Ask/Vol Version) first.")
    valid_input = False
else:
    missing_cols = [col for col in required_cols if col not in df_merged.columns]
    if missing_cols:
        print(f"Error: df_merged is missing required columns: {missing_cols}")
        valid_input = False
    else:
        print("Input df_merged found with required columns.")

# --- Feature Engineering ---
if valid_input:
    print("\nStarting Feature Engineering (Adding New Features)...")
    df_features = df_merged.copy()

    # --- 1. Calculate Target Variable ---
    print(f" - Calculating Target Variable: {TARGET_COL_BASE}")
    start_col = f'future_price_change_mid_{TARGET_HORIZON_START}'
    end_col = f'future_price_change_mid_{TARGET_HORIZON_END}'
    # Ensure target calculation only happens if base columns exist
    if start_col in df_features.columns and end_col in df_features.columns:
        df_features[TARGET_COL_BASE] = df_features[end_col] - df_features[start_col]
        df_features = df_features.dropna(subset=[TARGET_COL_BASE])
        print(f"   - Target calculated. Shape after NaN drop: {df_features.shape}")
        target_col_name = TARGET_COL_BASE # Set target name
    else:
        print(f"   - Error: Cannot calculate target '{TARGET_COL_BASE}'. Missing base columns.")
        target_col_name = None # Mark target as unavailable
        valid_input = False # Cannot proceed without target


if valid_input: # Proceed only if target was calculated
    # --- 2. Trade Specific & Aggression Features ---
    print(" - Creating Trade Specific & Aggression Features...")
    df_features['price_vs_mid'] = df_features['price'] - df_features['mid_price']
    # Spread calculated in Cell 6
    df_features['price_vs_mid_norm'] = (df_features['price_vs_mid'] / df_features['spread']).fillna(0).replace([np.inf, -np.inf], 0)

    def calculate_aggression(row):
        if pd.isna(row['bid_price_1']) or pd.isna(row['ask_price_1']) or row['spread'] <= 0: return 0
        if row['price'] >= row['ask_price_1']: return 1 # Aggressive Buy
        elif row['price'] <= row['bid_price_1']: return -1 # Aggressive Sell
        else: return 0 # Passive
    df_features['trade_aggression'] = df_features.apply(calculate_aggression, axis=1)

    # --- 3. Participant Identity Features ---
    print(" - Creating Participant Identity Features (One-Hot Encoding)...")
    all_participants = sorted(list(pd.unique(df_features[['buyer', 'seller']].values.ravel('K'))))
    df_features['buyer_orig'] = df_features['buyer'] # Keep original for interactions
    df_features['seller_orig'] = df_features['seller']
    df_features = pd.get_dummies(df_features, columns=['buyer'], prefix='b', prefix_sep='_')
    df_features = pd.get_dummies(df_features, columns=['seller'], prefix='s', prefix_sep='_')
    for p in all_participants:
        b_col, s_col = f'b_{p}', f's_{p}'
        if b_col not in df_features.columns: df_features[b_col] = 0
        if s_col not in df_features.columns: df_features[s_col] = 0

    # --- 4. Interaction Features ---
    print(" - Creating Interaction Features...")
    for b, s in INTERESTING_PAIRS:
        pair_col_name = f'pair_{b}_{s}'
        b_col, s_col = f'b_{b}', f's_{s}'
        if b_col in df_features.columns and s_col in df_features.columns:
             df_features[pair_col_name] = ((df_features['buyer_orig'] == b) & (df_features['seller_orig'] == s)).astype(int)
        else:
             df_features[pair_col_name] = 0
    df_features = df_features.drop(columns=['buyer_orig', 'seller_orig'])

    # --- 5. Market Context & Recent History Features ---
    print(" - Creating Market Context & Recent History Features...")
    df_features = df_features.sort_values(by=['symbol', 'timestamp'])
    gb = df_features.groupby('symbol')

    # --- 5a. Microstructure Features ---
    print("   - Calculating Microstructure Features...")
    # L1 Imbalance (handle division by zero)
    vol_sum = df_features['bid_volume_1'] + df_features['ask_volume_1']
    df_features['L1_imbalance'] = ((df_features['bid_volume_1'] - df_features['ask_volume_1']) / vol_sum).fillna(0).replace([np.inf, -np.inf], 0)
    # Weighted Mid-Price
    df_features['weighted_mid_price'] = ((df_features['bid_price_1'] * df_features['ask_volume_1'] + df_features['ask_price_1'] * df_features['bid_volume_1']) / vol_sum).fillna(df_features['mid_price']) # Fallback to mid_price

    # --- 5b. Price vs Range ---
    print("   - Calculating Price vs Range...")
    df_features[f'rolling_high_{ROLLING_WINDOW_LONG}'] = gb['mid_price'].transform(lambda x: x.rolling(ROLLING_WINDOW_LONG, 1).max())
    df_features[f'rolling_low_{ROLLING_WINDOW_LONG}'] = gb['mid_price'].transform(lambda x: x.rolling(ROLLING_WINDOW_LONG, 1).min())
    range_diff = df_features[f'rolling_high_{ROLLING_WINDOW_LONG}'] - df_features[f'rolling_low_{ROLLING_WINDOW_LONG}']
    df_features['price_vs_range'] = ((df_features['mid_price'] - df_features[f'rolling_low_{ROLLING_WINDOW_LONG}']) / range_diff).fillna(0.5).replace([np.inf, -np.inf], 0.5).clip(0,1)

    # --- 5c. Volatility Dynamics ---
    print("   - Calculating Volatility Dynamics...")
    df_features['mid_price_change'] = gb['mid_price'].diff().fillna(0) # Recalculate just in case
    df_features[f'rolling_vol_{ROLLING_WINDOW_SHORT}'] = gb['mid_price_change'].transform(lambda x: x.rolling(ROLLING_WINDOW_SHORT, 1).std()).fillna(0)
    df_features[f'rolling_vol_{ROLLING_WINDOW_LONG}'] = gb['mid_price_change'].transform(lambda x: x.rolling(ROLLING_WINDOW_LONG, 1).std()).fillna(0)
    df_features['volatility_trend'] = gb[f'rolling_vol_{ROLLING_WINDOW_SHORT}'].transform(lambda x: x.diff()).fillna(0) # Trend in short-term vol
    df_features['volatility_ratio'] = (df_features[f'rolling_vol_{ROLLING_WINDOW_SHORT}'] / df_features[f'rolling_vol_{ROLLING_WINDOW_LONG}']).fillna(1).replace([np.inf, -np.inf], 1)

    # --- 5d. Relative Trade Size ---
    print("   - Calculating Relative Trade Size...")
    df_features[f'rolling_avg_qty_{ROLLING_WINDOW_LONG}'] = gb['quantity'].transform(lambda x: x.rolling(ROLLING_WINDOW_LONG, 1).mean())
    df_features['qty_vs_rolling_avg'] = (df_features['quantity'] / df_features[f'rolling_avg_qty_{ROLLING_WINDOW_LONG}']).fillna(1).replace([np.inf, -np.inf], 1)
    # Qty vs L1 Vol (depends on trade aggression)
    def qty_vs_l1(row):
        if row['trade_aggression'] == 1 and row['ask_volume_1'] > 0: # Aggressive buy vs ask vol
            return row['quantity'] / row['ask_volume_1']
        elif row['trade_aggression'] == -1 and row['bid_volume_1'] > 0: # Aggressive sell vs bid vol
            return row['quantity'] / row['bid_volume_1'] # quantity is positive here
        else: # Passive or unknown aggression/volume
            return 0 # Or NaN? Let's use 0
    df_features['qty_vs_L1_vol'] = df_features.apply(qty_vs_l1, axis=1).fillna(0).replace([np.inf, -np.inf], 0)

    # --- 5e. Other Rolling Features ---
    df_features[f'rolling_trend_{ROLLING_WINDOW_SHORT}'] = gb['mid_price'].transform(lambda x: x.diff(ROLLING_WINDOW_SHORT-1)).fillna(0)
    df_features[f'rolling_trend_{ROLLING_WINDOW_LONG}'] = gb['mid_price'].transform(lambda x: x.diff(ROLLING_WINDOW_LONG-1)).fillna(0)
    df_features[f'rolling_aggression_{ROLLING_WINDOW_LONG}'] = gb['trade_aggression'].transform(lambda x: x.rolling(ROLLING_WINDOW_LONG, 1).sum()).fillna(0)


    # --- 6. Select Final Features and Target ---
    print(" - Selecting final features...")
    feature_cols = [
        # Trade Specifics
        'quantity', 'price_vs_mid', 'spread', 'price_vs_mid_norm', 'trade_aggression',
        'qty_vs_rolling_avg', 'qty_vs_L1_vol',
        # Market Context / Microstructure
        'mid_price_change', 'L1_imbalance', 'weighted_mid_price', 'price_vs_range',
        # Volatility / Trend
        f'rolling_vol_{ROLLING_WINDOW_SHORT}', f'rolling_vol_{ROLLING_WINDOW_LONG}',
        f'rolling_trend_{ROLLING_WINDOW_SHORT}', f'rolling_trend_{ROLLING_WINDOW_LONG}',
        'volatility_trend', 'volatility_ratio',
        # Rolling Aggression
        f'rolling_aggression_{ROLLING_WINDOW_LONG}',
    ]
    # Add OHE participant cols and specific pair cols
    ohe_cols = [col for col in df_features.columns if col.startswith('b_') or col.startswith('s_')]
    pair_cols = [col for col in df_features.columns if col.startswith('pair_')]
    feature_cols.extend(ohe_cols)
    feature_cols.extend(pair_cols)

    # Ensure all selected feature columns exist
    feature_cols = [col for col in feature_cols if col in df_features.columns]

    # Create final DataFrame for ML
    final_cols_for_ml = feature_cols + [target_col_name]
    df_ml = df_features[final_cols_for_ml].dropna().copy() # Drop rows with any NaNs

    print(f"   - Feature engineering complete. Final ML DataFrame shape: {df_ml.shape}")
    if not df_ml.empty:
        print(f"   - Features ({len(feature_cols)}): {feature_cols}")
        print(f"   - Target: {target_col_name}")
        print("\n--- ML DataFrame Head (Features + Target) ---")
        with pd.option_context('display.max_columns', None, 'display.width', 2000):
            print(df_ml.head())
    else:
        print("   - Final ML DataFrame is empty after NaN drop.")

else:
    print("\nSkipping Cell 15 due to validation errors or missing target.")

# df_ml is ready for splitting and training
# X = df_ml[feature_cols]
# y = df_ml[target_col_name]

In [ ]:
# Cell 16: Train and Evaluate Initial ML Model (Random Forest)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import time # To time training

# --- Configuration ---
TEST_SIZE = 0.2 # 20% of data for testing
RANDOM_STATE = 42 # For reproducible splits and model training
# RandomForest Parameters (Basic)
N_ESTIMATORS = 100 # Number of trees
MAX_DEPTH = 15     # Limit tree depth to prevent overfitting (adjust as needed)
N_JOBS = -1        # Use all available CPU cores
# ---

# --- Input Validation ---
valid_input = True
if 'df_ml' not in locals() or df_ml.empty:
    print(f"Error: df_ml DataFrame not found or is empty. Run Cell 15 first.")
    valid_input = False
elif 'target_col_name' not in locals() or not target_col_name:
     print(f"Error: target_col_name not defined. Run Cell 15 first.")
     valid_input = False
elif 'feature_cols' not in locals() or not feature_cols:
     print(f"Error: feature_cols not defined. Run Cell 15 first.")
     valid_input = False
elif target_col_name not in df_ml.columns:
     print(f"Error: Target column '{target_col_name}' not found in df_ml.")
     valid_input = False
elif not all(col in df_ml.columns for col in feature_cols):
     missing_f_cols = [col for col in feature_cols if col not in df_ml.columns]
     print(f"Error: Missing feature columns in df_ml: {missing_f_cols}")
     valid_input = False


# --- Model Training and Evaluation ---
if valid_input:
    print("\n--- Preparing Data for Model ---")
    X = df_ml[feature_cols]
    y = df_ml[target_col_name]
    print(f"Features shape (X): {X.shape}")
    print(f"Target shape (y): {y.shape}")

    # --- 1. Split Data (Standard Random Split - CAUTION for Time Series) ---
    # Note: For rigorous backtesting, a chronological split is essential.
    print(f"\n--- Splitting Data (Test Size: {TEST_SIZE*100:.0f}%, Random State: {RANDOM_STATE}) ---")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f"Train set size: {X_train.shape[0]} samples")
    print(f"Test set size: {X_test.shape[0]} samples")

    # --- 2. Initialize and Train Model ---
    print("\n--- Training RandomForestRegressor Model ---")
    # Initialize model with basic parameters
    rf_model = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS, # Use parallel processing
        oob_score=False # Out-of-bag score can be useful but adds time
    )

    start_time = time.time()
    rf_model.fit(X_train, y_train)
    end_time = time.time()
    print(f"Training completed in {end_time - start_time:.2f} seconds.")

    # --- 3. Make Predictions ---
    print("\n--- Making Predictions on Test Set ---")
    y_pred = rf_model.predict(X_test)

    # --- 4. Evaluate Model ---
    print("\n--- Evaluating Model Performance ---")
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"Mean Squared Error (MSE):      {mse:.4f}")
    print(f"Root Mean Squared Error (RMSE):{rmse:.4f}")
    print(f"Mean Absolute Error (MAE):     {mae:.4f}")
    print(f"R-squared (R²):                {r2:.4f}")

    # --- Baseline Comparison ---
    # Compare against predicting the mean of the training target
    baseline_pred = np.full_like(y_test, y_train.mean())
    baseline_mse = mean_squared_error(y_test, baseline_pred)
    baseline_rmse = np.sqrt(baseline_mse)
    baseline_mae = mean_absolute_error(y_test, baseline_pred)
    print("\n--- Baseline Performance (Predicting Mean) ---")
    print(f"Baseline RMSE:                 {baseline_rmse:.4f}")
    print(f"Baseline MAE:                  {baseline_mae:.4f}")
    if r2 > 0:
        print("Model performs better than baseline.")
    else:
        print("Model does NOT perform better than baseline (R² <= 0).")


    # --- 5. Feature Importance ---
    print("\n--- Feature Importances ---")
    importances = rf_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    print("Top 20 Features:")
    with pd.option_context('display.max_rows', 20):
        print(feature_importance_df.head(20))

    # Optional: Plot feature importances
    plt.figure(figsize=(10, 8))
    plt.barh(feature_importance_df['Feature'][:20], feature_importance_df['Importance'][:20])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title("Top 20 Feature Importances (RandomForest)")
    plt.gca().invert_yaxis() # Display most important at the top
    plt.tight_layout()
    plt.show()

    # --- 6. Actual vs. Predicted Plot ---
    print("\n--- Actual vs. Predicted Plot (Test Set Sample) ---")
    plt.figure(figsize=(8, 8))
    # Plot a sample if the test set is very large
    sample_size = min(1000, len(y_test))
    indices = np.random.choice(len(y_test), sample_size, replace=False)
    plt.scatter(y_test.iloc[indices], y_pred[indices], alpha=0.5, s=10)
    # Add y=x line
    lims = [min(plt.xlim()[0], plt.ylim()[0]), max(plt.xlim()[1], plt.ylim()[1])]
    plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0, label='Ideal (y=x)')
    plt.xlabel("Actual Delayed Change")
    plt.ylabel("Predicted Delayed Change")
    plt.title("Actual vs. Predicted (Sample)")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.axis('equal') # Ensure equal scaling for x and y axes
    plt.tight_layout()
    plt.show()

else:
    print("\nSkipping Cell 16 due to validation errors or missing data.")

In [ ]:
# Cell 17: Train Classification Models for Multiple Delayed Horizons

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import time

# --- Configuration ---
# Define the DELAYED windows to analyze: (start_horizon, end_horizon)
# These horizons MUST have corresponding future_price_change columns in df_merged (from Cell 6)
delayed_windows_to_analyze = [
    (100, 200),
    (100, 500),
    (200, 500), # Example: Change between t+200 and t+500
    (100, 1000),
    (200, 1000), # Example: Change between t+200 and t+1000
    (500, 1000), # Example: Change between t+500 and t+1000
    (100, 2500)
]

# ML Model Parameters
TEST_SIZE = 0.2
RANDOM_STATE = 42
# RandomForestClassifier Parameters (Basic)
N_ESTIMATORS = 100
MAX_DEPTH = 15 # Can tune this
N_JOBS = -1
CLASS_WEIGHT = 'balanced' # Useful if Up/Down moves are imbalanced
# ---

# --- Input Validation ---
valid_input = True
required_base_horizons = set()
for start_h, end_h in delayed_windows_to_analyze:
    required_base_horizons.add(start_h)
    required_base_horizons.add(end_h)

# Use df_features which has more rows before final dropna in Cell 15
if 'df_features' not in locals() or df_features.empty:
    print(f"Error: df_features DataFrame not found or is empty. Run Cell 15 first.")
    valid_input = False
elif 'feature_cols' not in locals() or not feature_cols:
     print(f"Error: feature_cols list not defined. Run Cell 15 first.")
     valid_input = False
else:
    # Check if all necessary base future_price_change columns exist in df_features
    missing_base_cols = []
    base_price_change_cols_needed = []
    for h in required_base_horizons:
        # Assuming we used mid-price changes in Cell 6
        col_name = f'future_price_change_mid_{h}'
        base_price_change_cols_needed.append(col_name)
        if col_name not in df_features.columns:
            missing_base_cols.append(col_name)

    if missing_base_cols:
        print(f"Error: df_features is missing required base columns from Cell 6: {missing_base_cols}")
        print(f"Ensure Cell 6 (Bid/Ask/Vol version) calculated mid-price horizons: {list(required_base_horizons)}")
        valid_input = False
    else:
        # Check if feature columns exist
        missing_f_cols = [col for col in feature_cols if col not in df_features.columns]
        if missing_f_cols:
             print(f"Error: df_features is missing required feature columns: {missing_f_cols}")
             valid_input = False
        else:
             print("Input df_features found with required columns.")

# --- Model Training Loop ---
results_summary = {} # Store metrics for each window

if valid_input:
    print(f"\n--- Starting Classification Training for {len(delayed_windows_to_analyze)} Delayed Windows ---")

    # Prepare Feature Matrix X - drop NaNs from features *before* the loop
    X_full = df_features[feature_cols].copy()
    print(f"Original feature rows: {len(X_full)}")
    X_full = X_full.dropna()
    print(f"Feature rows after dropping NaNs: {len(X_full)}")

    if X_full.empty:
        print("Error: Feature matrix X is empty after dropping NaNs. Cannot proceed.")
    else:
        # Loop through each defined delayed window
        for start_h, end_h in delayed_windows_to_analyze:
            window_label = f"{start_h}_{end_h}"
            print(f"\n===== Training for Window: t+{start_h} -> t+{end_h} =====")

            # --- 1. Calculate Delayed Change & Create Target y ---
            start_col = f'future_price_change_mid_{start_h}'
            end_col = f'future_price_change_mid_{end_h}'
            delayed_col = f'delayed_change_{window_label}'

            # Calculate on the original df_features, then align with X_full
            y_series_numeric = df_features[end_col] - df_features[start_col]

            # Create binary target: +1 if change >= 0, -1 if change < 0
            # Using >= 0 maps flat moves to the 'up' class
            y_series_binary = np.sign(y_series_numeric).replace(0, 1).rename(f'target_{window_label}')

            # Align y with the rows remaining in X_full (after dropping feature NaNs)
            y = y_series_binary.loc[X_full.index]

            # Check for NaNs in the target for this window *after* alignment
            if y.isna().any():
                print(f"Warning: NaNs found in target for window {window_label} after alignment. Dropping them.")
                valid_idx = y.dropna().index
                y = y.loc[valid_idx]
                X = X_full.loc[valid_idx] # Keep only corresponding features
                print(f"Aligned X shape: {X.shape}, y shape: {y.shape}")
            else:
                X = X_full # Use the full feature set if no target NaNs
                print(f"Target contains no NaNs. Using X shape: {X.shape}, y shape: {y.shape}")

            if X.empty or y.empty:
                 print(f"Error: No valid data remaining for window {window_label} after NaN handling.")
                 continue # Skip to next window

            # Check class balance
            class_counts = y.value_counts(normalize=True)
            print(f"Class distribution: \n{class_counts}")
            majority_class_fraction = class_counts.max()
            baseline_accuracy = majority_class_fraction # Accuracy if always predicting majority

            # --- 2. Train/Test Split ---
            print(f"Splitting data for window {window_label}...")
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y # Stratify helps with imbalance
            )
            print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

            # --- 3. Train RandomForestClassifier ---
            print(f"Training RandomForestClassifier for window {window_label}...")
            model = RandomForestClassifier(
                n_estimators=N_ESTIMATORS,
                max_depth=MAX_DEPTH,
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
                class_weight=CLASS_WEIGHT
            )
            start_time = time.time()
            model.fit(X_train, y_train)
            end_time = time.time()
            print(f"Training completed in {end_time - start_time:.2f} seconds.")

            # --- 4. Predict & Evaluate ---
            print(f"Evaluating model for window {window_label}...")
            y_pred = model.predict(X_test)
            y_proba = model.predict_proba(X_test)[:, 1] # Probability of class 1 (Up)

            accuracy = accuracy_score(y_test, y_pred)
            roc_auc = roc_auc_score(y_test, y_proba) # Requires probabilities

            print(f"\n--- Results for Window {window_label} ---")
            print(f"Baseline Accuracy (Predict Majority): {baseline_accuracy:.4f}")
            print(f"Model Accuracy:                     {accuracy:.4f}")
            print(f"ROC AUC Score:                      {roc_auc:.4f}")
            print("\nClassification Report:")
            print(classification_report(y_test, y_pred, target_names=['Down (-1)', 'Up/Flat (+1)']))
            print("\nConfusion Matrix:")
            cm = confusion_matrix(y_test, y_pred)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred Down', 'Pred Up'], yticklabels=['True Down', 'True Up'])
            plt.ylabel('Actual')
            plt.xlabel('Predicted')
            plt.title(f'Confusion Matrix - Window {window_label}')
            plt.show()

            # Store key results
            results_summary[window_label] = {
                'Accuracy': accuracy,
                'Baseline Acc': baseline_accuracy,
                'ROC AUC': roc_auc,
                'F1 (Down)': classification_report(y_test, y_pred, output_dict=True)['-1.0']['f1-score'],
                'F1 (Up)': classification_report(y_test, y_pred, output_dict=True)['1.0']['f1-score'],
            }

    # --- 5. Summarize Performance Across Windows ---
    print("\n\n===== Overall Performance Summary =====")
    summary_df = pd.DataFrame.from_dict(results_summary, orient='index')
    # Add comparison column
    summary_df['Acc vs Baseline'] = summary_df['Accuracy'] - summary_df['Baseline Acc']
    print(summary_df.sort_values(by='ROC AUC', ascending=False))

else:
    print("\nSkipping Cell 17 due to validation errors.")

In [ ]:
# Cell 18: Refined Analysis - Focusing on Pairs and Non-MarketMakers

import pandas as pd
import numpy as np

# --- Configuration ---
# Horizons displayed (should match those calculated in Cell 14)
delayed_windows_to_analyze = [
    (100, 200), (100, 500), (100, 1000), (100, 2500),
    (100, 5000), (100, 10000), (100, 25000), (100, 50000)
]
# Define likely Market Makers to exclude in Analysis B
market_makers_to_exclude = ['Paris, Caesar, Camilla']
# Ranking horizon for display
rank_window = delayed_windows_to_analyze[0] # e.g., (100, 200)
rank_col = f't_stat_{rank_window[0]}_{rank_window[1]}'
# ---

# --- Input Validation ---
if 'all_delayed_results_df' not in locals() or all_delayed_results_df.empty:
    print("Error: 'all_delayed_results_df' not found or is empty. Run Cell 14 first.")
    valid_input = False
elif rank_col not in all_delayed_results_df.columns:
     print(f"Error: Ranking column '{rank_col}' not found in results DataFrame.")
     valid_input = False
else:
    valid_input = True

if valid_input:
    print("--- Refined Analysis ---")

    # --- Analysis A: Focus on Pair & Self-Trade Interactions ---
    print("\n===== Analysis A: Focusing on Pair & Self-Trade Types =====")
    df_pairs_only = all_delayed_results_df[
        all_delayed_results_df['analysis_type'].isin(['Pair', 'Self-Trade'])
    ].copy()

    if df_pairs_only.empty:
        print("No 'Pair' or 'Self-Trade' type results found.")
    else:
        print(f"Filtered down to {len(df_pairs_only)} Pair/Self-Trade results.")
        # Rank this subset
        ranked_pairs_df = df_pairs_only.sort_values(by=rank_col, key=lambda col: col.abs(), ascending=False, na_position='last')

        # Prepare columns for display (same logic as Cell 14 display)
        id_cols = ['symbol', 'participant_or_pair', 'analysis_type', 'trade_count']
        display_cols_final_a = id_cols[:]
        rename_map_a = {}
        for start_h, end_h in delayed_windows_to_analyze:
            avg_col = f'avg_delayed_change_{start_h}_{end_h}'; tstat_col = f't_stat_{start_h}_{end_h}'
            if avg_col in ranked_pairs_df.columns: display_cols_final_a.append(avg_col)
            if tstat_col in ranked_pairs_df.columns: display_cols_final_a.append(tstat_col)
            if avg_col in ranked_pairs_df.columns: rename_map_a[avg_col] = f'Avg_{start_h}_{end_h}'
            if tstat_col in ranked_pairs_df.columns: rename_map_a[tstat_col] = f'T_{start_h}_{end_h}'

        ranked_df_display_a = ranked_pairs_df[display_cols_final_a].rename(columns=rename_map_a)
        final_print_cols_a = [rename_map_a.get(c, c) for c in display_cols_final_a]

        print(f"\n--- Top 20 Pair/Self-Trade Signals (Ranked by abs({rank_col})) ---")
        with pd.option_context('display.max_rows', 20, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.3f}'.format):
            print(ranked_df_display_a[final_print_cols_a].head(20))


    # --- Analysis B: Exclude Market Maker Interactions ---
    print(f"\n===== Analysis B: Excluding Market Makers: {market_makers_to_exclude} =====")

    # Filter out rows where participant_or_pair involves any excluded MM
    df_no_mm = all_delayed_results_df.copy()
    for mm in market_makers_to_exclude:
        # Exclude if MM is buyer OR seller in a pair/self-trade, OR if it's an MM->ALL / ALL->MM stat
        df_no_mm = df_no_mm[~df_no_mm['participant_or_pair'].str.contains(mm, na=False)]

    if df_no_mm.empty:
        print("No results found after excluding market makers.")
    else:
        print(f"Filtered down to {len(df_no_mm)} results excluding market makers.")
        # Rank this subset
        ranked_no_mm_df = df_no_mm.sort_values(by=rank_col, key=lambda col: col.abs(), ascending=False, na_position='last')

        # Prepare columns for display (reuse logic, apply to df_no_mm)
        id_cols = ['symbol', 'participant_or_pair', 'analysis_type', 'trade_count']
        display_cols_final_b = id_cols[:]
        rename_map_b = {}
        for start_h, end_h in delayed_windows_to_analyze:
            avg_col = f'avg_delayed_change_{start_h}_{end_h}'; tstat_col = f't_stat_{start_h}_{end_h}'
            if avg_col in ranked_no_mm_df.columns: display_cols_final_b.append(avg_col)
            if tstat_col in ranked_no_mm_df.columns: display_cols_final_b.append(tstat_col)
            if avg_col in ranked_no_mm_df.columns: rename_map_b[avg_col] = f'Avg_{start_h}_{end_h}'
            if tstat_col in ranked_no_mm_df.columns: rename_map_b[tstat_col] = f'T_{start_h}_{end_h}'

        ranked_df_display_b = ranked_no_mm_df[display_cols_final_b].rename(columns=rename_map_b)
        final_print_cols_b = [rename_map_b.get(c, c) for c in display_cols_final_b]

        print(f"\n--- Top 20 Non-MarketMaker Signals (Ranked by abs({rank_col})) ---")
        with pd.option_context('display.max_rows', 20, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.3f}'.format):
            print(ranked_df_display_b[final_print_cols_b].head(20))

else:
    print("Skipping Cell 18 due to validation errors or missing input DataFrame.")

In [ ]:
# Cell 19: Visualize Participant's Trades, Position, PnL & Stats Across Assets

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# --- Configuration ---
# <<< SET THE PARTICIPANT TO VISUALIZE >>>
participant_to_visualize = 'Paris'
# ---

# --- Input Validation ---
valid_input = True
if 'df_prices' not in locals() or df_prices.empty:
    print("Error: df_prices not found or is empty. Run Cell 1 first.")
    valid_input = False
elif 'df_trades' not in locals() or df_trades.empty:
    print("Error: df_trades not found or is empty. Run Cell 1 first.")
    valid_input = False
elif not all(col in df_prices.columns for col in ['product', 'timestamp', 'mid_price', 'bid_price_1', 'ask_price_1']): # Added bid/ask for context
     print(f"Error: df_prices is missing required columns.")
     valid_input = False
elif not all(col in df_trades.columns for col in ['symbol', 'timestamp', 'buyer', 'seller', 'price', 'quantity']):
     print(f"Error: df_trades is missing required columns.")
     valid_input = False

# --- Main Logic ---
if valid_input:
    print(f"\n--- Generating Trade Plots & Stats for Participant: {participant_to_visualize} ---")

    # --- 1. Find all symbols traded by this participant ---
    participant_trades_all = df_trades[
        (df_trades['buyer'] == participant_to_visualize) |
        (df_trades['seller'] == participant_to_visualize)
    ]
    traded_symbols = sorted(participant_trades_all['symbol'].unique())

    if not traded_symbols:
        print(f"No trades found for participant '{participant_to_visualize}' in the dataset.")
    else:
        print(f"Found {len(traded_symbols)} assets traded by {participant_to_visualize}: {traded_symbols}")

        # --- 2. Loop through each traded symbol ---
        for i, symbol in enumerate(traded_symbols):
            print(f"\n===== Analyzing Symbol {i+1}/{len(traded_symbols)}: {symbol} =====")

            # --- Filter Data for this Symbol ---
            symbol_prices = df_prices[df_prices['product'] == symbol].copy()
            symbol_prices = symbol_prices.sort_values('timestamp').drop_duplicates(subset=['timestamp'], keep='last')

            participant_symbol_trades = participant_trades_all[
                participant_trades_all['symbol'] == symbol
            ].copy().sort_values('timestamp')

            # Separate buys and sells
            buy_trades = participant_symbol_trades[participant_symbol_trades['buyer'] == participant_to_visualize]
            sell_trades = participant_symbol_trades[participant_symbol_trades['seller'] == participant_to_visualize]

            # --- Check if data exists ---
            pnl_calculated = False
            pnl_timeline = pd.DataFrame()

            if symbol_prices.empty:
                print(f"Warning: No price data found for symbol '{symbol}'. Skipping plot.")
                continue
            if participant_symbol_trades.empty:
                 print(f"Warning: No trades found for {participant_to_visualize} in {symbol}. Skipping plot.")
                 continue

            # --- 3. Calculate Summary Statistics ---
            print(f"\n--- Summary Stats for {participant_to_visualize} in {symbol} ---")
            if not buy_trades.empty:
                print(f"  Buys ({len(buy_trades)} trades):")
                print(f"    - Mean Volume:   {buy_trades['quantity'].mean():,.2f}")
                print(f"    - Median Volume: {buy_trades['quantity'].median():,.1f}")
                print(f"    - Mean Price:    {buy_trades['price'].mean():,.2f}")
            else:
                print("  Buys: None")

            if not sell_trades.empty:
                print(f"  Sells ({len(sell_trades)} trades):")
                print(f"    - Mean Volume:   {sell_trades['quantity'].mean():,.2f}")
                print(f"    - Median Volume: {sell_trades['quantity'].median():,.1f}")
                print(f"    - Mean Price:    {sell_trades['price'].mean():,.2f}")
            else:
                print("  Sells: None")

            # --- 4. Calculate Running PnL (Similar to Cell 5 logic) ---
            print("\n - Calculating PnL timeline...")
            try:
                participant_symbol_trades['position_change'] = participant_symbol_trades.apply(
                    lambda row: row['quantity'] if row['buyer'] == participant_to_visualize else -row['quantity'], axis=1)
                participant_symbol_trades['cumulative_position'] = participant_symbol_trades['position_change'].cumsum()
                participant_symbol_trades['cash_flow'] = participant_symbol_trades.apply(
                    lambda row: -row['quantity'] * row['price'] if row['buyer'] == participant_to_visualize else row['quantity'] * row['price'], axis=1)
                participant_symbol_trades['cumulative_cash_flow'] = participant_symbol_trades['cash_flow'].cumsum()

                trade_times = participant_symbol_trades['timestamp'].unique()
                price_times = symbol_prices['timestamp'].unique()
                all_timestamps = np.union1d(price_times, trade_times)
                all_timestamps = np.sort(all_timestamps)

                pnl_timeline = pd.DataFrame(index=all_timestamps)
                pnl_timeline.index.name = 'timestamp'
                trades_indexed = participant_symbol_trades.set_index('timestamp')[['cumulative_position', 'cumulative_cash_flow']]
                prices_indexed = symbol_prices.set_index('timestamp')[['mid_price']]
                pnl_timeline = pnl_timeline.join(trades_indexed, how='outer')
                pnl_timeline = pnl_timeline.join(prices_indexed, how='outer')

                pnl_timeline['cumulative_position'] = pnl_timeline['cumulative_position'].ffill().fillna(0)
                pnl_timeline['cumulative_cash_flow'] = pnl_timeline['cumulative_cash_flow'].ffill().fillna(0)
                pnl_timeline['mid_price'] = pnl_timeline['mid_price'].ffill().bfill().fillna(0)

                pnl_timeline['inventory_value'] = pnl_timeline['cumulative_position'] * pnl_timeline['mid_price']
                pnl_timeline['mtm_pnl'] = pnl_timeline['cumulative_cash_flow'] + pnl_timeline['inventory_value']
                pnl_calculated = True
                print("   - PnL calculation successful.")
            except Exception as e:
                print(f"   - Error calculating PnL: {e}")
                pnl_calculated = False


            # --- 5. Create Plotly Figure ---
            print(" - Generating plot...")
            fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                              vertical_spacing=0.03,
                              subplot_titles=(f'Prices & {participant_to_visualize}\'s Trades',
                                              f'{participant_to_visualize}\'s Position',
                                              f'{participant_to_visualize}\'s MtM PnL'))

            # == Plot 1: Prices and Trades ==
            fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['mid_price'], mode='lines', name='Mid Price', line=dict(color='grey', width=1.5, dash='dash'), opacity=0.9, legendgroup='1'), row=1, col=1)
            # Optional: Add Bid/Ask lines
            # fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['bid_price_1'], mode='lines', name='Best Bid', line=dict(color='blue', width=1), opacity=0.7, legendgroup='1'), row=1, col=1)
            # fig.add_trace(go.Scatter(x=symbol_prices['timestamp'], y=symbol_prices['ask_price_1'], mode='lines', name='Best Ask', line=dict(color='red', width=1), opacity=0.7, legendgroup='1'), row=1, col=1)

            # Buy Markers
            if not buy_trades.empty:
                fig.add_trace(go.Scatter(x=buy_trades['timestamp'], y=buy_trades['price'], mode='markers', marker=dict(color='lime', size=7, symbol='triangle-up', line=dict(color='black', width=1)), name=f'Buys', legendgroup='1', hovertemplate='<b>Buy</b><br>Timestamp: %{x}<br>Price: %{y:,.2f}<br>Quantity: %{customdata[0]}<extra></extra>', customdata=buy_trades[['quantity']]), row=1, col=1)
            # Sell Markers
            if not sell_trades.empty:
                fig.add_trace(go.Scatter(x=sell_trades['timestamp'], y=sell_trades['price'], mode='markers', marker=dict(color='red', size=7, symbol='triangle-down', line=dict(color='black', width=1)), name=f'Sells', legendgroup='1', hovertemplate='<b>Sell</b><br>Timestamp: %{x}<br>Price: %{y:,.2f}<br>Quantity: %{customdata[0]}<extra></extra>', customdata=sell_trades[['quantity']]), row=1, col=1)

            # == Plot 2: Position ==
            if not participant_symbol_trades.empty:
                 fig.add_trace(go.Scatter(x=participant_symbol_trades['timestamp'], y=participant_symbol_trades['cumulative_position'], mode='lines', line_shape='hv', name='Position', line=dict(color='purple', width=2), legendgroup='2', hovertemplate='Timestamp: %{x}<br>Position: %{y}<extra></extra>'), row=2, col=1)
            fig.add_hline(y=0, line_dash="dot", line_color="grey", line_width=1, row=2, col=1)

            # == Plot 3: PnL ==
            if pnl_calculated and not pnl_timeline.empty:
                 fig.add_trace(go.Scatter(x=pnl_timeline.index, y=pnl_timeline['mtm_pnl'], mode='lines', line_shape='linear', name='MtM PnL', line=dict(color='orange', width=2), legendgroup='3', hovertemplate='Timestamp: %{x}<br>PnL: %{y:,.0f}<extra></extra>'), row=3, col=1)
            fig.add_hline(y=0, line_dash="dot", line_color="grey", line_width=1, row=3, col=1)

            # --- Layout Updates ---
            fig.update_layout(
                title=f'Analysis for {participant_to_visualize} in {symbol}', # Main title
                height=800, # Adjust height if needed
                legend_traceorder="reversed",
                hovermode='x unified'
            )
            fig.update_yaxes(title_text='Price', row=1, col=1); fig.update_yaxes(title_text="Position", row=2, col=1); fig.update_yaxes(title_text="MtM PnL", row=3, col=1)
            fig.update_xaxes(showticklabels=False, row=1, col=1); fig.update_xaxes(showticklabels=False, row=2, col=1); fig.update_xaxes(title_text="Timestamp", row=3, col=1)

            fig.show() # Display plot for this symbol
            print(f"--- End Analysis for {symbol} ---")

else:
    print("\nSkipping Cell 19 due to validation errors.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Filter Camilla's BUY trades on JAMS ---
camilla_buys = df_trades[
    (df_trades["symbol"] == "JAMS") &
    (df_trades["buyer"] == "Camilla")
].copy()

camilla_buys = camilla_buys.sort_values("timestamp")

# Define quantity buckets
bins = [0, 5, 10, 20, np.inf]
labels = ["1-5", "6-10", "11-20", "21+"]

camilla_buys["size_bucket"] = pd.cut(camilla_buys["quantity"], bins=bins, labels=labels, right=True)

# Analyze price movement after each buy by size bucket
bucket_movements = {label: [] for label in labels}

for _, row in camilla_buys.iterrows():
    ts = row["timestamp"]
    size_label = row["size_bucket"]
    
    future_prices = df_prices[
        (df_prices["product"] == "JAMS") &
        (df_prices["timestamp"] >= ts)
    ].sort_values("timestamp")
    
    if not future_prices.empty:
        initial = future_prices.iloc[0]["mid_price"]
        future = future_prices.head(20)["mid_price"].reset_index(drop=True)
        deltas = future - initial
        bucket_movements[size_label].append(deltas)

# Plot average response for each bucket
plt.figure(figsize=(12, 6))

for label in labels:
    df = pd.DataFrame(bucket_movements[label])
    if not df.empty:
        mean_response = df.mean()
        std_response = df.std()
        plt.plot(mean_response, label=f"{label} Qty (n={len(df)})")
        plt.fill_between(mean_response.index, mean_response - std_response, mean_response + std_response, alpha=0.15)

plt.axhline(0, linestyle="--", color="gray")
plt.title("Mid Price Response After Camilla Buys JAMS — By Trade Size")
plt.xlabel("Ticks After Camilla Buy")
plt.ylabel("Mid Price Change")
plt.legend(title="Buy Size Bucket")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 20: Analyze Subsequent Moves When Immediate Drop Signal Fails

import pandas as pd
import numpy as np

# --- Configuration ---
EVENT_DESCRIPTION = "Pablo Buys PICNIC_BASKET1 (Immediate Drop Fails)"
EVENT_FILTER = {
    "buyer": "Peter",
    "seller": None,
    "symbol": "VOLCANIC_ROCK",
    "timestamp_start": None,
    "timestamp_end": None
}
# Define the IMMEDIATE window where the drop was EXPECTED (t to t+100)
SIGNAL_WINDOW_HORIZON = 100
# Define SUBSEQUENT windows to check for reversal/strength (relative to t)
SUBSEQUENT_WINDOWS = [
    (100, 200), (100, 500), (100, 1000), (200, 500), (200, 1000), (500, 1000)
]
min_failed_signal_count = 5
# ---

# --- Input Validation ---
valid_input = True
required_base_horizons = set()
required_base_horizons.add(SIGNAL_WINDOW_HORIZON) # Need the immediate horizon outcome
for start_h, end_h in SUBSEQUENT_WINDOWS:
    required_base_horizons.add(start_h)
    required_base_horizons.add(end_h)

if 'df_merged' not in locals() or df_merged.empty:
    print(f"Error: df_merged not found or is empty. Run Cell 6 first.")
    valid_input = False
else:
    missing_base_cols = []
    base_price_change_cols_needed = []
    for h in required_base_horizons:
        col_name = f'future_price_change_mid_{h}'
        base_price_change_cols_needed.append(col_name)
        if col_name not in df_merged.columns: missing_base_cols.append(col_name)
    if missing_base_cols:
        print(f"Error: df_merged is missing required base columns: {missing_base_cols}")
        valid_input = False
    else: print("Input df_merged found with required columns.")

# --- Helper: T-Stat vs Zero ---
def calculate_t_stat_vs_zero(mean, std, n):
    if pd.isna(mean) or pd.isna(std) or std == 0 or n < 2: return np.nan
    se = std / np.sqrt(n); t_stat = mean / se
    return t_stat

# --- Main Analysis Logic ---
if valid_input:
    print(f"\nAnalyzing subsequent moves after '{EVENT_DESCRIPTION}' signal failed.")
    print(f"Expected drop window: t to t+{SIGNAL_WINDOW_HORIZON}")

    # --- 1. Filter df_merged for the specific trigger event ---
    event_trades = df_merged.copy()
    print(" - Applying event filters...")
    # ... (apply filters based on EVENT_FILTER) ...
    if EVENT_FILTER.get("buyer"): event_trades = event_trades[event_trades['buyer'] == EVENT_FILTER["buyer"]]
    if EVENT_FILTER.get("seller"): event_trades = event_trades[event_trades['seller'] == EVENT_FILTER["seller"]]
    if EVENT_FILTER.get("symbol"): event_trades = event_trades[event_trades['symbol'] == EVENT_FILTER["symbol"]]
    if EVENT_FILTER.get("timestamp_start"): event_trades = event_trades[event_trades['timestamp'] >= EVENT_FILTER["timestamp_start"]]
    if EVENT_FILTER.get("timestamp_end"): event_trades = event_trades[event_trades['timestamp'] < EVENT_FILTER["timestamp_end"]]

    if event_trades.empty:
        print(f"\nNo trades found matching the specified event criteria.")
    else:
        print(f"Found {len(event_trades)} total trades matching the event criteria.")

        # --- 2. Identify Signal Failures ---
        # Use the pre-calculated immediate change column
        signal_outcome_col = f'future_price_change_mid_{SIGNAL_WINDOW_HORIZON}'
        print(f" - Identifying failures based on column: {signal_outcome_col}")

        # Signal FAILED if the actual change was >= 0 (it didn't drop)
        failed_signals = event_trades[event_trades[signal_outcome_col] >= 0].copy()
        success_signals = event_trades[event_trades[signal_outcome_col] < 0].copy()

        if failed_signals.empty:
            print(f"\nNo instances found where the expected drop signal failed for '{EVENT_DESCRIPTION}'.")
        else:
            print(f"Found {len(failed_signals)} instances where the signal failed (actual change >= 0).")
            print(f"(Signal succeeded {len(success_signals)} times).")

            # --- 3. Calculate SUBSEQUENT delayed changes for FAILED signals ---
            print(" - Calculating subsequent delayed changes for FAILED signal instances...")
            subsequent_results = []
            # Group failed signals by symbol (might only be one if symbol filter was set)
            for symbol, group_df in failed_signals.groupby('symbol'):
                if len(group_df) < min_failed_signal_count:
                    print(f"   - Skipping {symbol}: Only {len(group_df)} failed instances (min: {min_failed_signal_count}).")
                    continue

                symbol_stats = {'symbol': symbol, 'failed_count': len(group_df)}
                valid_subsequent_window = False
                for start_h, end_h in SUBSEQUENT_WINDOWS:
                    sub_start_col = f'future_price_change_mid_{start_h}'
                    sub_end_col = f'future_price_change_mid_{end_h}'
                    sub_delayed_col = f'subsequent_change_{start_h}_{end_h}'

                    if sub_start_col in group_df.columns and sub_end_col in group_df.columns:
                        # Calculate the change *within* the subsequent window
                        group_df[sub_delayed_col] = group_df[sub_end_col] - group_df[sub_start_col]

                        avg_change = group_df[sub_delayed_col].mean()
                        std_change = group_df[sub_delayed_col].std()
                        count = group_df[sub_delayed_col].count()
                        t_stat = calculate_t_stat_vs_zero(avg_change, std_change, count)

                        symbol_stats[f'AvgSub_{start_h}_{end_h}'] = avg_change
                        # symbol_stats[f'StdSub_{start_h}_{end_h}'] = std_change # Optional
                        symbol_stats[f'TSub_{start_h}_{end_h}'] = t_stat
                        if count >= min_failed_signal_count : valid_subsequent_window = True
                    else:
                        print(f"   - Warning: Skipping subsequent window {start_h}-{end_h} for {symbol}, missing base columns.")

                if valid_subsequent_window: subsequent_results.append(symbol_stats)

            # --- 4. Display Results ---
            if not subsequent_results:
                print(f"\nNo symbols met the minimum count ({min_failed_signal_count}) for failed signal analysis.")
            else:
                subsequent_results_df = pd.DataFrame(subsequent_results)
                print(f"\n--- Average SUBSEQUENT Price Change After '{EVENT_DESCRIPTION}' Signal FAILED ---")
                # (Keep display logic as before, maybe rank by TSub_100_200 or TSub_100_500)
                id_cols = ['symbol', 'failed_count']
                stat_cols = []
                rename_map = {}
                for start_h, end_h in SUBSEQUENT_WINDOWS:
                    avg_col = f'AvgSub_{start_h}_{end_h}'; tstat_col = f'TSub_{start_h}_{end_h}'
                    if avg_col in subsequent_results_df.columns: stat_cols.append(avg_col)
                    if tstat_col in subsequent_results_df.columns: stat_cols.append(tstat_col)
                    if avg_col in subsequent_results_df.columns: rename_map[avg_col] = f'Avg_{start_h}_{end_h}'
                    if tstat_col in subsequent_results_df.columns: rename_map[tstat_col] = f'T_{start_h}_{end_h}'
                display_cols = id_cols + sorted(stat_cols)
                display_cols = [c for c in display_cols if c in subsequent_results_df.columns]
                subsequent_results_df_display = subsequent_results_df[display_cols].rename(columns=rename_map)
                final_print_cols = [rename_map.get(c, c) for c in display_cols]
                rank_col_sub = f'T_{SUBSEQUENT_WINDOWS[0][0]}_{SUBSEQUENT_WINDOWS[0][1]}' # Rank by first subsequent window T-stat
                if rank_col_sub in subsequent_results_df_display.columns:
                     subsequent_results_df_display = subsequent_results_df_display.sort_values(by=rank_col_sub, key=lambda col: col.abs(), ascending=False, na_position='last')
                with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 2000, 'display.float_format', '{:,.3f}'.format):
                    print(subsequent_results_df_display[final_print_cols])
                print("\n--- Interpretation Notes ---") # ... (notes) ...

else:
     print("Skipping Cell 20 due to validation errors.")